# Avaliação automática de sentenças em português

Este documento examina onze técnicas, uma por seção. Cada seção apresenta o **fundamento matemático**, um
**exemplo mínimo** executado, a **leitura do resultado** e as condições de
**aplicabilidade**.


**Ortografia**

| # | técnica | o que mede |
|---:|---|---|
| 1 | [Léxico com distância de edição](#t01-lexico) | pertinência ao léxico e forma corrigida |

**Gramática**

| # | técnica | o que mede |
|---:|---|---|
| 2 | [Traços morfossintáticos (parser)](#t02-tracos) | concordância, com o token e o traço apontados |
| 3 | [Regras escritas à mão (LanguageTool)](#t03-regras) | violação de norma previamente codificada, com a justificativa |
| 4 | [SLOR](#t04-slor) | aceitabilidade global, descontada a frequência das palavras |
| 5 | [PLL](#t05-pll) | aceitabilidade sob modelo mascarado, dentro do par mínimo |

**Estrutura**

| # | técnica | o que mede |
|---:|---|---|
| 6 | [Teste do núcleo predicativo](#t06-raiz) | presença de predicado; distinção entre oração e fragmento |
| 7 | [Índices de complexidade sintática](#t07-complexidade) | encaixamento e número de orações |
| 8 | [Comprimento de dependência e não-projetividade](#t08-dependencia) | distância entre palavras ligadas; arcos que se cruzam |

---

**Síntese**

| # | técnica | o que mede |
|---:|---|---|
| 9 | [A pipeline — as técnicas em cadeia](#pipeline) | as técnicas selecionadas, em encadeamento |

---

**Semântica**

| # | técnica | o que mede |
|---:|---|---|
| 10 | [Surpresa por token](#t10-surpresa) | improbabilidade local, posição a posição |
| 11 | [Plausibilidade do verbo mascarado](#t11-verbo) | adequação do verbo à posição que ocupa |
| 12 | [Embeddings](#t12-embeddings) | proximidade temática entre palavras |


In [1]:
%conda install -c conda-forge -y spacy=3.8 spacy-model-pt_core_news_md pyspellchecker language_tool_python openjdk nltk numpy requests

# --- corpus para o modelo de n-grama ---------------------------
import nltk
nltk.download("mac_morpho")


Channels:
 - conda-forge
Platform: win-64
Solving environment: ...working... done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html

[nltk_data] Downloading package mac_morpho to
[nltk_data]     C:\Users\Matheus\AppData\Roaming\nltk_data...
[nltk_data]   Package mac_morpho is already up-to-date!


True

<a id="t01-lexico"></a>

## Léxico com distância de edição

A técnica não emprega modelo nem generalização: opera sobre um **conjunto finito de
formas atestadas**, cada uma associada à sua frequência no corpus. Decompõe-se em dois
predicados encadeados.

### 1. Detecção — pertinência a um conjunto

Dado o léxico $D$:

$$\mathrm{erro}(w) = \begin{cases} 0, & w \in D \\ 1, & w \notin D \end{cases}$$

O predicado é binário e exato: a forma consta ou não consta do conjunto, sem gradação,
limiar ou margem de incerteza. Constitui o único veredito exato entre as técnicas
examinadas neste documento.

### 2. Correção — o máximo de frequência na vizinhança

Seja $C_k(w)$ o conjunto das formas do léxico situadas a exatamente $k$ edições de $w$:

$$C_k(w) = \{\, c \in D \;:\; d_L(w,c) = k \,\}$$

A busca é **escalonada**: a distância 2 só é consultada quando a distância 1 não devolve
candidato algum.

$$C(w) = \begin{cases} C_1(w), & C_1(w) \neq \emptyset \\[4pt] C_2(w), & C_1(w) = \emptyset \end{cases}$$

Entre os candidatos admitidos, a frequência determina a escolha:

$$\hat{w} = \arg\max_{c \,\in\, C(w)} f(c)$$

O escalonamento não constitui otimização de eficiência: ele altera o conjunto submetido
ao $\arg\max$. Havendo qualquer candidato a uma edição, nenhuma forma a duas edições é
considerada.

### A distância de edição

$d_L(a,b)$ é o número mínimo de inserções, remoções, substituições e **transposições
de caracteres adjacentes** que transformam $a$ em $b$, dado pela recorrência

$$d_{i,j} = \min \begin{cases}
d_{i-1,j} + 1 & \text{(remoção)} \\
d_{i,j-1} + 1 & \text{(inserção)} \\
d_{i-1,j-1} + [\,a_i \neq b_j\,] & \text{(substituição)} \\
d_{i-2,j-2} + 1 & \text{(transposição, se } a_i = b_{j-1} \text{ e } a_{i-1} = b_j\text{)}
\end{cases}$$

com $d_{i,0} = i$ e $d_{0,j} = j$. O termo $[\,a_i \neq b_j\,]$ assume valor 1 quando os
caracteres diferem e 0 quando coincidem.

A quarta linha da recorrência não é acessória: é ela que situa `natvia` a **uma** edição
de `nativa`. Sob a métrica restrita às três primeiras operações, a permuta de dois
caracteres adjacentes custaria 2, `nativa` ficaria fora de $C_1$ e o $\arg\max$ recairia
sobre `naevia`. A métrica que incorpora a transposição denomina-se
**Damerau-Levenshtein**, e é a implementada pelo `pyspellchecker`.

A frequência $f$ intervém exclusivamente na segunda etapa; a detecção é dela
independente.


In [2]:
import os
import requests
from spellchecker import SpellChecker

URL = ("https://api.github.com/repos/hermitdave/FrequencyWords/contents/"
       "content/2018/pt_br/pt_br_full.txt")
CACHE = "pt_br_full.txt"


if not os.path.exists(CACHE):
    r = requests.get(URL, headers={"Accept": "application/vnd.github.raw"}, timeout=120)
    r.raise_for_status()
    with open(CACHE, "w", encoding="utf-8") as f:
        f.write(r.text)

frequencias = {}
for linha in open(CACHE, encoding="utf-8"):
    partes = linha.split()
    if len(partes) == 2 and partes[1].isdigit() and int(partes[1]) >= 20:
        frequencias[partes[0]] = int(partes[1])

lexico = SpellChecker(language=None)
lexico.word_frequency.load_json(frequencias)

def revisar(frase):
    print(frase)
    desconhecidas = lexico.unknown(lexico.split_words(frase))
    if not desconhecidas:
        print("Nenhum erro ortográfico encontrado.")
        return
    for p in desconhecidas:
        print(f"Erro ortográfico: {p}")
        candidatos = lexico.candidates(p)
        if not candidatos:
            print("      sem candidatos no léxico\n")
            continue
        print(f"candidatos, por frequência:")
        for c in sorted(candidatos, key=lambda c: -frequencias[c]):
            print(f"{c:<12}{frequencias[c]:>9}")
        print(f"      correção: {lexico.correction(p)}\n")

revisar("Há três dias o incêndio atingiu uma área de mata natvia.")
revisar("Há três dias o incêndio atingiu uma área de mata nativa.")
revisar("A três dias o incêndio atingiu uma área de mata nativa.")


Há três dias o incêndio atingiu uma área de mata natvia.
Erro ortográfico: natvia
candidatos, por frequência:
nativa            912
naevia            103
natia              77
latvia             23
      correção: nativa

Há três dias o incêndio atingiu uma área de mata nativa.
Nenhum erro ortográfico encontrado.
A três dias o incêndio atingiu uma área de mata nativa.
Nenhum erro ortográfico encontrado.


### Aplicação das fórmulas aos valores observados

A técnica encadeia dois predicados, e o código submete três sentenças. Percorrem-se a
seguir as duas fórmulas com os valores obtidos.

---

#### Primeiro predicado — detecção

$$\mathrm{erro}(w) = \begin{cases} 0, & w \in D \\ 1, & w \notin D \end{cases}$$

A avaliação é feita **sobre uma forma por vez**, contra o conjunto. Para a primeira
sentença:

| $w$ | $w \in D$ ? | $\mathrm{erro}(w)$ |
|---|:---:|:---:|
| `Há` | sim | 0 |
| `três` | sim | 0 |
| `dias` | sim | 0 |
| `o` | sim | 0 |
| `incêndio` | sim | 0 |
| `atingiu` | sim | 0 |
| `uma` | sim | 0 |
| `área` | sim | 0 |
| `de` | sim | 0 |
| `mata` | sim | 0 |
| `natvia` | **não** | **1** |

São onze avaliações do predicado, uma delas positiva. Cabe caracterizar o que a fórmula
**não** consulta: nem a posição da forma na sentença, nem as formas vizinhas, nem a
frequência. O predicado se reduz a $w \in D$, razão pela qual este é o único veredito
exato do documento.

---

#### Segundo predicado — correção

Apenas a forma detectada prossegue para a segunda etapa. Aplica-se primeiro o
escalonamento:

$$C(w) = \begin{cases} C_1(w), & C_1(w) \neq \emptyset \\[4pt] C_2(w), & C_1(w) = \emptyset \end{cases}$$

```
C1(natvia):     949 variações distintas  ->  4 existem em D
C2(natvia):  417.548 variações distintas ->  nunca consultadas
```

$$C_1(\texttt{natvia}) \neq \emptyset \quad\Longrightarrow\quad C(\texttt{natvia}) = C_1(\texttt{natvia})$$

O segundo ramo da expressão não é avaliado. Existem 48 formas do léxico a **exatamente**
duas edições de `natvia`, e **nenhuma delas é submetida ao $\arg\max$** — a fórmula as
exclui por construção, não por inferioridade de frequência. (O conjunto que o código
gera para a distância 2 contém 52 formas conhecidas ao todo, das quais quatro são as
próprias vizinhas de distância 1, já admitidas na etapa anterior.)

Procede-se então ao critério de desempate entre os quatro candidatos admitidos:

$$\hat{w} = \arg\max_{c \,\in\, C(w)} f(c)$$

| $c$ | $d_L$ | operação sobre `natvia` | $f(c)$ | $P(c) = f/N$ | $\ln P$ |
|---|:---:|---|---:|---:|---:|
| **nativa** | 1 | transpõe `v` e `i` | 912 | $2{,}15 \times 10^{-6}$ | $-13{,}05$ |
| naevia | 1 | troca `t` $\to$ `e` | 103 | $2{,}43 \times 10^{-7}$ | $-15{,}23$ |
| natia | 1 | remove o `v` | 77 | $1{,}82 \times 10^{-7}$ | $-15{,}52$ |
| latvia | 1 | troca `n` $\to$ `l` | 23 | $5{,}42 \times 10^{-8}$ | $-16{,}73$ |

$$\hat{w} = \texttt{nativa}$$

A coluna $d_L$ é constante: os quatro candidatos equidistam da forma detectada, e três
operações distintas da recorrência estão representadas — transposição, remoção e
substituição. Uma vez que $d_L$ não estabelece ordenação entre eles, a decisão recai
**integralmente sobre $f$**: o $\arg\max$ não opera como critério auxiliar, mas como
único critério.

As contagens são brutas; com $N = 424{.}015{.}479$,

$$P(\texttt{nativa}) = \frac{912}{424{.}015{.}479} = 2{,}15 \times 10^{-6}$$

isto é, uma ocorrência a cada 465 mil tokens. Os quatro candidatos distribuem-se por
mais de uma ordem de grandeza: a razão `nativa`/`naevia` é 8,9, e `nativa`/`latvia`
atinge 40. A decisão não é marginal.

Em escala logarítmica, os valores vão de $-13{,}05$ a $-16{,}73$, com separação de 3,7
nats. Convém fixar essa escala desde já: é nela que operam as técnicas probabilísticas
subsequentes, e é precisamente este termo que o SLOR subtrai.

---

#### As duas sentenças restantes — a fórmula avaliando para zero

O código submete mais duas sentenças, decididas pelo mesmo predicado:

| frase | $w$ examinada | $w \in D$ ? | $\mathrm{erro}(w)$ |
|---|---|:---:|:---:|
| …de mata **nativa**. | `nativa` | sim | 0 |
| **A** três dias… | `a` | sim | 0 |

Nenhuma das duas produz achado, embora se encontrem em situações opostas: a segunda
sentença está correta; a terceira deveria apresentar *Há*, e não *A*.

A fórmula não dispõe de meios para distingui-las. A forma `a` pertence a $D$ com
$f = 8{.}992{.}538$ ocorrências, situando-se entre as mais frequentes do português;
logo $\mathrm{erro}(\texttt{a}) = 0$ **por definição**, e nenhum léxico alternativo
altera esse resultado — um léxico mais extenso apenas reforçaria a pertinência.

Este par é retomado na seção 3, em que uma regra escrita manualmente detecta
precisamente o que o léxico não alcança.

---

#### O conjunto resultante

$$E_{\text{orto}}(s_1) = \{\,\texttt{natvia}\,\}
\qquad
E_{\text{orto}}(s_2) = \varnothing
\qquad
E_{\text{orto}}(s_3) = \varnothing$$

Três sentenças produzem um único achado, acompanhado da localização (a forma) e da
correção ($\hat{w}$). Trata-se da saída mais informativa entre as onze técnicas:
nenhuma outra devolve a forma que **deveria** figurar no texto.


### Geração cega e filtragem lexical

O gerador produz as 949 sequências situadas a uma edição da forma detectada — `aatvia`,
`batvia`, `natvi`, `natviaa` —, das quais a grande maioria não corresponde a forma
alguma do português. O léxico atua como filtro e elimina 945 delas. O termo "candidato"
é, nesse contexto, impreciso: o algoritmo não busca formas semelhantes, mas **enumera
exaustivamente o espaço de edições e submete cada elemento ao teste de pertinência**.

### Onde funciona

O primeiro exemplo exercita as duas etapas. A forma `natvia` não consta do léxico:
detecção exata, sem margem de incerteza. A vizinhança a uma edição contém quatro formas
reais, todas igualmente admissíveis do ponto de vista lexical. Sem o critério de
frequência, a saída se limitaria a um conjunto de quatro alternativas; é $f$ que
converte um diagnóstico em correção.

### Onde falha, e por quê

O par seguinte contém um erro inequívoco — a forma correta é `há`, e não `a` — e a
técnica não produz achado em nenhuma das duas versões. A causa foi estabelecida acima:
`a` é forma do português, pertence a $D$, e o predicado se esgota nesse fato.

A distinção pertinente é entre **erro de grafia** (a forma não existe) e **erro de
escolha** (a forma existe, mas não é a adequada ao contexto). A etapa que determina o
veredito avalia uma forma por vez, isoladamente, e `a`, tomada isoladamente, não
apresenta desvio. O que caracteriza o erro é a relação com a expressão temporal que a
forma encabeça e com o verbo no pretérito — informação de natureza sintática, ausente
do léxico.

Este é o limite superior do método, e a justificativa para as demais seções deste
documento.

### Quando usar

- **Como primeira etapa, invariavelmente.** É a única técnica examinada cujo veredito é
  **exato** — a forma consta ou não consta, sem limiar nem gradação — e cujo custo
  computacional é desprezível.
- **Quando se exige a correção, e não apenas a localização do erro.** Nenhuma das demais
  técnicas aqui tratadas devolve a forma pretendida; limitam-se a sinalizar anomalia.
- **Como pré-requisito das medidas de frequência.** Forma fora do vocabulário compromete
  qualquer estimativa probabilística; recomenda-se suspender o cálculo do SLOR na
  presença de forma desconhecida na sentença.

### Quando não usar

- **Para erro de escolha lexical.** `a` por `há`, `mais` por `mas`, `mal` por `mau`:
  todas essas formas pertencem a $D$, donde $\mathrm{erro}(w) = 0$ **por definição**. A
  limitação não é do recurso lexical — um léxico mais extenso apenas reforçaria a
  pertinência.
- **Sob a expectativa de que a distância determine a correção.** No exemplo, os quatro
  candidatos situam-se todos a uma edição: a coluna $d_L$ é constante e a decisão cabe a
  $f$ isoladamente. Se o corpus de frequências não representar o domínio do texto, a
  correção incorreta é emitida com a mesma confiança que a correta.
- **Sobre nome próprio, termo técnico ou estrangeirismo.** Tais formas não constam do
  léxico e são detectadas como erro. Trata-se de falso positivo por construção, não por
  defeito de implementação.
- **Sem conhecimento da extensão e da procedência do léxico.** A cobertura de $D$
  **constitui** a definição operacional de erro; substituir a lista altera o veredito
  sem alteração alguma no código.


<a id="t02-tracos"></a>

## Traços morfossintáticos sobre a árvore de dependências (Parser)

A concordância não constitui heurística passível de aproximação: define-se como
**igualdade de traços entre dois nós ligados na árvore sintática**. A técnica transcreve
literalmente essa definição.

### O objeto

O analisador associa a cada token $t$ três informações: a classe gramatical, o **núcleo**
$h(t)$ do qual ele depende, e um mapa de traços

$$\mu_\tau(t) \in \{\text{valor}, \bot\}$$

em que $\tau$ designa o traço (`Number`, `Gender`, `Person`) e $\bot$ indica ausência.
Numerosos tokens, entre os quais preposições e advérbios, não portam número nem gênero.

A árvore é o conjunto de arcos $A = \{(t,\, r,\, h(t))\}$, sendo $r$ a função sintática
(`det`, `amod`, `nsubj`, ...).

### A regra

Nem todo arco impõe concordância, e cada tipo impõe traços distintos. Seja $T(r)$ o
conjunto de traços cuja coincidência o arco $r$ exige:

$$T(r) = \begin{cases}
\{\text{Number},\ \text{Gender}\}, & r \in \{\texttt{det},\ \texttt{amod},\ \texttt{acl}\},\ \ h \text{ é NOUN/PROPN} \\[4pt]
\{\text{Number},\ \text{Person}\}, & r \in \{\texttt{nsubj},\ \texttt{nsubj:pass}\},\ \ h \text{ é VERB/AUX} \\[4pt]
\varnothing, & \text{caso contrário}
\end{cases}$$

O conjunto de erros da sentença $s$ é

$$E(s) = \bigl\{\, (t,\,h,\,\tau) \;:\; (t,r,h) \in A,\ \ \tau \in T(r),\ \
\mu_\tau(t) \neq \bot,\ \ \mu_\tau(h) \neq \bot,\ \ \mu_\tau(t) \neq \mu_\tau(h) \,\bigr\}$$

e a sentença é aceita quando $E(s) = \varnothing$.

### Três decisões inscritas na fórmula

**A guarda $\mu_\tau \neq \bot$ é obrigatória.** Na sua ausência, todo token desprovido
de número seria computado como discordante de seu núcleo, e a técnica acusaria a
sentença integralmente.

**O terceiro caso, $T(r) = \varnothing$, previne o falso positivo mais frequente.** Em
*as flores do jardim*, `jardim` liga-se a `flores` por `nmod`, e substantivos ligados
por `nmod` **não** estabelecem concordância entre si. A inclusão de `nmod` faria a
técnica acusar toda sentença dotada de complemento nominal.

**A inclusão de `acl` não é acessória.** Em *os dados coletados*, o particípio
`coletados` recebe a função `acl` e a classe VERB, e não `amod`; sua omissão suprimiria
integralmente a verificação de concordância de particípio.

A saída não é um escore: identifica o token responsável, o traço violado e o termo com
o qual ele diverge.


In [3]:
import spacy

nlp = spacy.load("pt_core_news_md")

NOMINAL = {"det","amod","acl"}
VERBAL = {"nsubj","nsubj:pass"}

def concordancia(frase):
    doc = nlp(frase)
    print(frase)
    achou = False
    for t in doc:
        h = t.head
        if t.dep_ in NOMINAL and h.pos_ in ("NOUN", "PROPN"):
            tracos = ("Number", "Gender")
        elif t.dep_ in VERBAL and h.pos_ in ("VERB", "AUX"):
            tracos = ("Number", "Person")
        else:
            continue
        for traco in tracos:
            a, b = t.morph.get(traco), h.morph.get(traco)
            if a and b and a != b:
                achou = True
                print(f"   -> {traco}: {t.text}({a[0]}) x {h.text}({b[0]})  [{t.dep_}]")
    if not achou:
        print("   --> Nenhuma discordância encontrada.")
    print()


concordancia("Há três dias o incêndio atingiu uma áreas de mata nativa.")
concordancia("Há três dias o incêndio atingiu a área inteiro.")

for t in nlp("Há três dias o incêndio atingiu a área inteiro."):
    print(f"{t.text:<10}{t.pos_:<7}{t.dep_:<8}{t.morph}")

Há três dias o incêndio atingiu uma áreas de mata nativa.
   -> Number: uma(Sing) x áreas(Plur)  [det]

Há três dias o incêndio atingiu a área inteiro.
   --> Nenhuma discordância encontrada.

Há        VERB   case    Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin
três      NUM    nummod  NumType=Card
dias      NOUN   obl     Gender=Masc|Number=Plur
o         DET    det     Definite=Def|Gender=Masc|Number=Sing|PronType=Art
incêndio  NOUN   nsubj   Gender=Masc|Number=Sing
atingiu   VERB   ROOT    Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin
a         DET    det     Definite=Def|Gender=Fem|Number=Sing|PronType=Art
área      NOUN   obj     Gender=Fem|Number=Sing
inteiro   ADJ    amod    Gender=Fem|Number=Sing
.         PUNCT  punct   


### Aplicação das fórmulas aos valores observados

O código submete duas sentenças. A fórmula é idêntica em ambas; varia apenas o insumo.

---

#### Primeira sentença — o arco discordante

```
Há três dias o incêndio atingiu uma áreas de mata nativa.
```

A primeira etapa consiste em $T(r)$ determinar quais arcos são admitidos. Dos dez arcos
da sentença, apenas quatro são examinados:

| arco $(t, r, h)$ | $r$ | $h$ é | $T(r)$ |
|---|---|---|---|
| (`Há`, case, `dias`) | case | — | $\varnothing$ |
| (`três`, nummod, `dias`) | nummod | — | $\varnothing$ |
| (`dias`, obl, `atingiu`) | obl | — | $\varnothing$ |
| (`o`, det, `incêndio`) | det | NOUN | {Number, Gender} |
| (`incêndio`, nsubj, `atingiu`) | nsubj | VERB | {Number, Person} |
| (`uma`, det, `áreas`) | det | NOUN | {Number, Gender} |
| (`áreas`, obj, `atingiu`) | obj | — | $\varnothing$ |
| (`de`, case, `mata`) | case | — | $\varnothing$ |
| (`mata`, nmod, `áreas`) | nmod | — | $\varnothing$ |
| (`nativa`, amod, `mata`) | amod | NOUN | {Number, Gender} |

Seis arcos recaem no terceiro caso da expressão. O arco de `mata` corresponde
precisamente ao caso antecipado no fundamento: substantivo ligado a substantivo por
`nmod` **não** estabelece concordância, e sua inclusão faria a técnica acusar toda
sentença dotada de complemento nominal.

Em seguida, cada arco admitido desdobra-se nos traços especificados por $T(r)$:

| arco | $\tau$ | $\mu_\tau(t)$ | $\mu_\tau(h)$ | veredito |
|---|---|:---:|:---:|---|
| (`o`, det, `incêndio`) | Number | Sing | Sing | iguais |
| | Gender | Masc | Masc | iguais |
| (`incêndio`, nsubj, `atingiu`) | Number | Sing | Sing | iguais |
| | Person | $\bot$ | 3 | **comparação suprimida** |
| (`uma`, det, `áreas`) | Number | **Sing** | **Plur** | $\in E(s)$ |
| | Gender | Fem | Fem | iguais |
| (`nativa`, amod, `mata`) | Number | Sing | Sing | iguais |
| | Gender | Fem | Fem | iguais |

$$E(s) = \bigl\{\, (\texttt{uma},\ \texttt{áreas},\ \texttt{Number}) \,\bigr\}$$

A quarta linha exemplifica a guarda $\mu_\tau \neq \bot$. O substantivo `incêndio` não
porta traço de pessoa, ao passo que o verbo apresenta `Person=3`. A comparação é
**suprimida**: o resultado não é de igualdade, mas de ausência de termos a comparar. Sem
essa guarda, todo sujeito nominal do português seria computado como discordante de seu
verbo quanto à pessoa.

A saída, por sua vez, não é um escore. A tripla
$(\texttt{uma}, \texttt{áreas}, \texttt{Number})$ identifica o token responsável, o termo
com o qual ele diverge e o traço violado; a esses se soma o arco `det` que os liga —
quatro informações passíveis de apresentação ao autor do texto.

---

#### Segunda sentença — a fórmula avaliando para vazio

```
Há três dias o incêndio atingiu a área inteiro.
```

O desvio é inequívoco: a forma esperada é *a área inteira*, e não *inteiro*. A fórmula,
contudo, não produz achado.

| arco | $\tau$ | $\mu_\tau(t)$ | $\mu_\tau(h)$ | veredito |
|---|---|:---:|:---:|---|
| (`o`, det, `incêndio`) | Number | Sing | Sing | iguais |
| | Gender | Masc | Masc | iguais |
| (`incêndio`, nsubj, `atingiu`) | Number | Sing | Sing | iguais |
| | Person | $\bot$ | 3 | comparação suprimida |
| (`a`, det, `área`) | Number | Sing | Sing | iguais |
| | Gender | Fem | Fem | iguais |
| (`inteiro`, amod, `área`) | Number | Sing | Sing | iguais |
| | Gender | **Fem** | **Fem** | iguais |

$$E(s) = \varnothing$$

A última linha da tabela é decisiva. A forma escrita é `inteiro`, de gênero masculino, e
ainda assim $\mu_{\text{Gender}}(\texttt{inteiro}) = \texttt{Fem}$.

Nenhuma etapa do percurso falhou: o arco `amod` existe e está corretamente atribuído,
$T(r)$ o admitiu, ambas as guardas foram satisfeitas e a comparação foi efetuada.
Comparou-se `Fem` com `Fem`, obteve-se igualdade e concluiu-se pela concordância.

O que a fórmula recebeu não foi $\mu$, e sim $\hat\mu$ — estimativa que já incorporava o
erro.

A causa reside em que **os traços não são lidos da forma escrita, mas preditos do
contexto** por um modelo estatístico:

$$\hat\mu_\tau(t) = \arg\max_{v} \; P\!\left(v \mid \text{contexto}\right)$$

O contexto é *"a área ___"*, no qual o valor esperado é o feminino. O modelo emitiu o
valor mais **plausível**, e não o mais **fiel** à forma escrita; o desvio foi eliminado
antes que a regra pudesse examiná-lo.

A formulação de $E(s)$ pressupõe que $\mu$ seja função da forma escrita. Não o é: é
$\hat\mu$, uma estimativa — e a técnica não supera o mais frágil de seus insumos.

---

### A oposição de fundo

O modelo empregado declara `morph_acc` de **95,6%** e `dep_las` de **86,0%**, tendo sido
treinado sobre o *UD Portuguese Bosque* v2.8: um conjunto de traços incorreto a cada 23
tokens.

O valor, contudo, oculta o aspecto essencial. Os componentes foram treinados em **texto
correto** e otimizados para produzir a análise mais plausível, ao passo que um detector
de erros requer a análise mais fiel. As duas exigências são opostas, e a oposição é
estrutural: não se resolve por volume de dados nem por aumento de capacidade do modelo,
uma vez que o que difere é o objetivo do treinamento.

Trata-se do limite de fundo de toda técnica dependente de analisador treinado.

### Quando usar

- **Quando se exige a localização do erro.** A técnica devolve a tripla $(t, h, \tau)$
  acrescida do arco — quatro informações acionáveis, passíveis de apresentação ao autor
  do texto. Nenhuma outra técnica examinada fornece localização.
- **Para concordância nominal e verbal.** Não há heurística envolvida: a regra é a
  transcrição literal da definição — igualdade de traços entre nós ligados na árvore.
- **Como etapa de baixo custo, anterior a qualquer modelo.** O tempo de execução é da
  ordem de milissegundos, sem requisito de *download* ou treinamento.

### Quando não usar

- **Sob a suposição de que $\mu$ seja lido da forma escrita.** Trata-se de $\hat\mu$,
  **predito do contexto**: `inteiro` recebe `Gender=Fem` porque em *"a área ___"* o valor
  esperado é o feminino, e o desvio é eliminado antes que a regra possa examiná-lo.
- **Em relações não cobertas por $T(r)$** — regência, colocação, tempo verbal —, nas
  quais não há traço a comparar. A ampliação indiscriminada de $T(r)$ produz falso
  positivo: substantivos ligados por `nmod`, como em *as flores do jardim*, não
  concordam entre si.
- **Sem conhecimento do `morph_acc` e do `dep_las` do modelo empregado.** Os limites
  observados pertencem ao recurso, e não ao método: a substituição do modelo altera o
  resultado sem alteração da regra.
- **Como detector isolado sobre texto mal representado pelo analisador.** Este foi
  treinado em texto correto e otimizado para a análise mais plausível, tendendo a
  normalizar precisamente o desvio que se pretende detectar.


<a id="t03-regras"></a>

## Regras escritas à mão (LanguageTool)

A gramática normativa encontra-se codificada em compêndios e gramáticas de referência.
Esta técnica não realiza inferência: cada regra é transcrita manualmente como um **padrão sobre a
sequência anotada**, e o verificador busca casamentos.

### O objeto

A sentença é tokenizada e anotada, constituindo uma sequência de triplas

$$s = \bigl\langle (w_1, \ell_1, p_1),\ (w_2, \ell_2, p_2),\ \ldots,\ (w_n, \ell_n, p_n) \bigr\rangle$$

em que $w_i$ designa a forma escrita, $\ell_i$ o lema e $p_i$ a etiqueta morfológica.

### A regra

Cada regra $r$ consiste em um padrão $\pi_r$ — expressão sobre essas triplas, análoga a
uma expressão regular, porém operando sobre atributos linguísticos em lugar de
caracteres. A regra dispara nas posições em que o padrão casa:

$$D_r(s) = \{\, i \;:\; \pi_r \text{ casa em } s \text{ a partir da posição } i \,\}$$

e a saída da técnica é a união sobre a **gramática** $R$, o conjunto finito de regras
elaboradas para o idioma:

$$D(s) = \bigcup_{r \in R} \bigl\{ (i,\, r) \;:\; i \in D_r(s) \bigr\}$$

Cada disparo carrega o **identificador da regra** $r$, o que permite exibir a
justificativa em texto redigido — recurso ausente nas demais técnicas examinadas.

### A propriedade determinante

$R$ é **finito e de elaboração manual**. Daí decorre, sem gradação intermediária:

$$\text{erro} \notin \bigcup_{r \in R} \mathcal{L}(\pi_r) \;\Longrightarrow\; \text{invisível}$$

em que $\mathcal{L}(\pi_r)$ designa o conjunto de construções reconhecidas pelo padrão
$r$. Não há generalização, grau de confiança ou aproximação: na ausência de regra
correspondente, o desvio não é detectado. A cobertura da técnica **equivale** ao trabalho
humano investido no idioma — trabalho de natureza voluntária, distribuído de forma
acentuadamente desigual entre as línguas.


In [4]:
import os
import language_tool_python
from language_tool_python import download_lt

download_lt.LATEST_VERSION = "6.6"

os.environ["JAVA_TOOL_OPTIONS"] = "-Dstdout.encoding=UTF-8 -Dstderr.encoding=UTF-8 -Dfile.encoding=UTF-8"
tool = language_tool_python.LanguageTool("pt-BR")


def regras(frase):
    print(frase)
    achados = tool.check(frase)
    if not achados:
        print("   -> nenhuma regra disparou\n")
        return
    for m in achados:
        trecho = frase[m.offset:m.offset + m.errorLength]
        print(f"   -> regra {m.ruleId}")
        print(f"      trecho: {trecho!r}   sugestões: {m.replacements[:4]}")
        print(f"      {m.message}\n")

regras("A três dias o incêndio atingiu uma área de mata nativa.")
regras("Há três dias os incêndios atingiu uma área de mata nativa.")

A três dias o incêndio atingiu uma área de mata nativa.
   -> regra CONFUSÃO_À_HÁ
      trecho: 'A'   sugestões: ['Há']
      Para expressões no passado ou se pretende utilizar o verbo “haver”, substitua por “há”.

Há três dias os incêndios atingiu uma área de mata nativa.
   -> nenhuma regra disparou



### Aplicação das fórmulas aos valores observados

Esta técnica apresenta uma particularidade ausente nas duas anteriores, e ela condiciona
a forma como o percurso pode ser exposto. Convém enunciá-la previamente.

$$D(s) = \bigcup_{r \in R} \bigl\{ (i,\, r) \;:\; i \in D_r(s) \bigr\}$$

O quantificador percorre $R$, que reúne **2.270 regras** para o português, elaboradas ao
longo de anos por colaboradores voluntários. Diferentemente de $D$ na seção 1 ou de
$T(r)$ na seção 2, **o percurso não é exibível**: não é praticável apresentar 2.270
linhas nem inspecionar o padrão $\pi_r$ de cada regra.

Exibe-se apenas o resultado: quais regras casaram, e em que posição.

---

#### Primeira sentença — a regra que casa

```
A três dias o incêndio atingiu uma área de mata nativa.
```

$$D(s) = \bigl\{\, (0,\ \texttt{CONFUSÃO\_À\_HÁ}) \,\bigr\}$$

O achado comporta diversos campos, cada qual veiculando informação distinta:

| campo | valor | o que é |
|---|---|---|
| `ruleId` | `CONFUSÃO_À_HÁ` | **qual** regra casou — o $r$ da fórmula |
| `offset` | 0 | **onde** casou — o $i$ da fórmula |
| `errorLength` | 1 | a extensão do casamento |
| trecho | `'A'` | o que $\pi_r$ cobriu |
| `replacements` | `['Há']` | a correção que a regra carrega |
| `message` | *"Para expressões no passado ou se pretende utilizar o verbo haver, substitua por há."* | a **norma, em português** |

O par $(i, r)$ constitui a localização: $i = 0$ indica a posição no texto, e
`errorLength = 1` delimita a extensão do trecho — um caractere, o `A` inicial.

A última linha corresponde ao que nenhuma outra técnica fornece. A seção 1 devolveu
`natvia -> nativa`, um par de formas. A seção 2 devolveu
$(\texttt{uma}, \texttt{áreas}, \texttt{Number})$, uma tripla de símbolos. Esta devolve
**um enunciado em português que explicita a norma**, redigido conjuntamente com o padrão.

Cabe registrar que se trata **exatamente da sentença não detectada na seção 1**. Ali,
$\mathrm{erro}(\texttt{a}) = 0$ porque `a` pertence a $D$ com $f = 8{.}992{.}538$, e
nenhum léxico alternativo alteraria esse resultado. Aqui, uma regra de elaboração manual
detecta o caso — não por inferência, mas porque foi codificada explicitamente a
observação de que artigo seguido de numeral e unidade de tempo corresponde com
frequência ao verbo *haver* grafado incorretamente.

---

#### Segunda sentença — o conjunto vazio não explicável

```
Há três dias os incêndios atingiu uma área de mata nativa.
```

$$D(s) = \varnothing$$

Sujeito no plural, verbo no singular: o desvio de concordância verbal mais elementar do
português. Nenhuma das 2.270 regras casa.

O percurso encontra aqui seu limite, e o limite é da técnica. Nas seções 1 e 2 foi
possível exibir *a razão* de a fórmula não ter acusado: a tabela apresentava `a ∈ D` com
sua frequência, ou $\hat\mu_{\text{Gender}}(\texttt{inteiro}) = \texttt{Fem}$. O valor
responsável pelo conjunto vazio permanecia visível.

Aqui não há tabela a exibir. A fórmula estabelece $\bigcup_{r \in R}$, e determinar a
causa do conjunto vazio exigiria verificar que **nenhum** dos 2.270 padrões casa —
verificação não realizável a partir da saída.

Resta inspecionar o próprio pacote de regras. O resultado é mais informativo do que a
hipótese de ausência de regra: ela **está escrita**.

```
pt/grammar.xml:9136
   <rulegroup id='GENERAL_VERB_AGREEMENT_ERRORS'
              name="Concordância Verbal: Geral">
```

O grupo reúne vinte e dois padrões e **dispara** efetivamente, ainda que não sobre a
sentença em exame:

```
Eles fez o trabalho.        -> GENERAL_VERB_AGREEMENT_ERRORS   ['Eles fizeram', 'Ele fez']
Nós vai ao mercado.         -> GENERAL_VERB_AGREEMENT_ERRORS   ['Nós vamos', 'Nos vai']

Os incêndios atingiu uma área.   -> nenhuma regra
Os meninos comeu o bolo.         -> nenhuma regra
```

O corte é identificável: os padrões estão ancorados em **sujeito pronominal**. `Eles` e
`Nós` constituem lista fechada de formas, comportável em um padrão. Sujeito **nominal**
(`os incêndios`, `os meninos`) exigiria casar determinante, substantivo e verbo por
traço, através de um sintagma de comprimento arbitrário — operação que $\pi_r$ não
realiza.

Este é o enunciado preciso da limitação, e ele é mais forte que o anterior: não se trata
de ausência de trabalho humano sobre o fenômeno, mas de que **a linguagem de padrões não
alcança a construção**, ainda que a regra esteja escrita e ativa. Onde o sujeito se
reduz a uma lista, a regra opera; onde constitui sintagma, não.

---

#### A propriedade, agora com valores

$$\text{erro} \notin \bigcup_{r \in R} \mathcal{L}(\pi_r) \;\Longrightarrow\; \text{invisível}$$

E $R$ equivale ao trabalho humano investido no idioma. Comparam-se a seguir os valores
do pacote:

| idioma | regras | grupos de regras |
|---|---:|---:|
| inglês | 5.628 | 980 |
| alemão | 4.851 | 1.195 |
| **português** | **2.270** | **249** |

*(contagem de `<rule>` e `<rulegroup>` em `grammar.xml` e nos arquivos de variante, na
LanguageTool 6.6 — a mesma instalação empregada na célula anterior.)*

O português ocupa a última posição em ambas as colunas, sendo a segunda a mais
significativa. Um `rulegroup` reúne as variantes de um mesmo fenômeno, de modo que a
contagem de grupos aproxima o **número de fenômenos distintos** efetivamente cobertos:
249 contra 980 e 1.195 — cerca de um **quarto** do que alcança o inglês e um
**quinto** do que alcança o alemão.

O que falta não é volume, mas **variedade de construções cobertas**. E a concordância
verbal com sujeito nominal — a mais elementar do português — está entre as construções
não cobertas, a despeito dos vinte e dois padrões escritos para o fenômeno.

---

### Uma observação adicional

A regra `CONFUSÃO_À_HÁ` **não dispara** quando a expressão temporal se encontra distante
do verbo. Na formulação *"o incêndio atingiu uma área de mata nativa a três dias"*, com o
adjunto em posição final, a ferramenta não produz achado, embora o desvio seja o mesmo.

O padrão $\pi_r$ casa uma vizinhança, não um significado. O deslocamento do constituinte
retira a construção do alcance do padrão sem retirá-la do alcance da norma — distinção
entre reconhecer uma **sequência** e interpretar uma **relação**.

### Complementaridade, não ordenação

Confrontam-se a seguir as três técnicas examinadas até aqui sobre as variantes do mesmo
corpus:

| variante | Léxico (T1) | Traços (T2) | Regras (T3) |
|---|:---:|:---:|:---:|
| `natvia` (grafia) | **detecta** | não | **detecta** |
| `uma áreas` (conc. nominal) | não | **detecta** | não |
| `incêndios atingiu` (conc. verbal) | não | **detecta** | não |
| `a três dias` (escolha) | não | não | **detecta** |

Não há coluna dominante. A questão pertinente não é qual técnica é superior, mas **qual
fenômeno cada uma cobre**.

As três em conjunto ainda não detectam a variante mais relevante do corpus:

```
Há três dias o incêndio cantou uma área de mata nativa.      T1 —   T2 —   T3 —
```

A árvore é bem formada, todas as formas constam do léxico e nenhuma regra casa. O
defeito consiste em violação de **restrição de seleção** — *incêndio* não é argumento
admissível de *cantar* —, fenômeno inacessível às três técnicas.

A variante embaralhada merece registro à parte, por ilustrar risco de outra natureza. O
LanguageTool **dispara** sobre ela em três ocasiões:

```
nativa uma incêndio há mata de o área três atingiu dias.
   -> UPPERCASE_SENTENCE_START
   -> GENERAL_GENDER_AGREEMENT_ERRORS
   -> ERRO_DE_CONCORDNCIA_DO_GÉNERO_MASCULINO_O
```

Nenhum dos três acusa a ordenação. O primeiro assinala a minúscula inicial; os dois
restantes apontam concordâncias que existem apenas porque o embaralhamento justapôs
formas de modo fortuito. **A ferramenta identificou corretamente a existência de desvio
e caracterizou-o incorretamente em sua totalidade.**

O caso é ilustrativo: disparo não equivale a diagnóstico. A contagem de regras acionadas
é pouco informativa quando nenhuma delas descreve o defeito efetivo.

### Quando usar

- **Quando se exige a explicitação da norma ao autor do texto.** É a única técnica que
  fornece o identificador da regra acompanhado de enunciado redigido em português. Uma
  medida numérica devolveria "3,7"; o analisador devolveria
  `Number: uma(Sing) x áreas(Plur)`; apenas esta enuncia **a razão**.
- **Para desvios frequentes, bem delimitados e previamente codificados** — as confusões
  de escolha como *à/há*, *mal/mau*, *mas/mais*, inacessíveis ao léxico porque ambas as
  formas existem.
- **Em combinação, nunca isoladamente.** A tabela acima evidencia que sua cobertura é
  complementar à do analisador sintático.

### Quando não usar

- **Sob expectativa de cobertura.** $R$ é finito e de elaboração manual: na ausência de
  regra correspondente, o desvio não é detectado. Não há grau de confiança nem
  aproximação — a resposta é binária.
- **Em português, sob expectativa da mesma cobertura do inglês ou do alemão.** A questão
  não é de volume — o português dispõe de 2.270 regras contra 4.851 do alemão —, mas de
  variedade: 249 grupos contra 1.195. A regra de concordância verbal existe, porém seus
  padrões alcançam apenas sujeito pronominal.
- **Interpretando a ausência de disparo como aprovação.** *Os incêndios atingiu* não
  aciona regra alguma e é agramatical. A ausência de achado indica ausência de regra, e
  não ausência de desvio.
- **Sobre texto literário ou técnico, sem revisão dos disparos.** Construções
  deliberadamente afastadas da norma produzem falso positivo.


<a id="t04-slor"></a>

## SLOR — Syntactic Log-Odds Ratio (razão sintática de log-chances)

Uma sentença malformada é improvável sob um modelo de linguagem, o que sugeriria a
leitura direta de $\ln P(s)$. A probabilidade bruta, entretanto, não distingue **raro**
de **incorreto**: uma sentença correta de vocabulário técnico é improvável por conter
formas improváveis, e não por má construção.

O SLOR corrige essa indistinção descontando a contribuição atribuível à frequência
isolada das formas e normalizando pelo comprimento:

$$\boxed{\;\mathrm{SLOR}(s) = \frac{\ln P_{\mathrm{LM}}(s) \;-\; \ln P_{\mathrm{uni}}(s)}{|s|}\;}$$

### Os dois termos

O primeiro corresponde ao modelo **sensível ao contexto**; no caso, um bigrama:

$$\ln P_{\mathrm{LM}}(s) = \sum_{i=1}^{|s|} \ln P(w_i \mid w_{i-1})$$

O segundo corresponde ao mesmo texto sob modelo que **ignora a ordem** — amostragem
independente de formas:

$$\ln P_{\mathrm{uni}}(s) = \sum_{i=1}^{|s|} \ln P(w_i)$$

A subtração isola a contribuição da *estrutura*. Caso a ordem não veiculasse informação,
os dois termos coincidiriam e o SLOR seria nulo.

**O sinal é interpretável**, propriedade infrequente entre as medidas examinadas:

$$\mathrm{SLOR}(s) > 0 \;\Rightarrow\; \text{a ordem acrescenta probabilidade}$$
$$\mathrm{SLOR}(s) < 0 \;\Rightarrow\; \text{a sequência é anti-estrutural — o modelo prevê melhor sob amostragem independente das formas}$$

### As estimativas

As probabilidades derivam de contagens no corpus, sem recurso a modelo treinado:

$$P(w) = \frac{c(w) + 1}{N + V} \qquad\qquad
P(w_i \mid w_{i-1}) = \lambda\,\frac{c(w_{i-1}, w_i)}{c(w_{i-1})} \;+\; (1-\lambda)\,P(w_i)$$

Os termos $+1$ e $+V$ constituem a suavização de Laplace, que impede que uma forma nunca
observada anule a estimativa. A interpolação por $\lambda$ resolve o problema análogo no
bigrama: quando o par $(w_{i-1}, w_i)$ não foi observado, a estimativa **recua** para a
distribuição unigrama.

Essa recuperação é a origem do modo de falha caracterizado adiante.


In [5]:
import math, re
from collections import Counter
from nltk.corpus import mac_morpho

INICIO = "<s>"

sentencas = [[p.lower() for p in s if p.isalpha()] for s in mac_morpho.sents()]
sentencas = [s for s in sentencas if s]

palavras = [w for s in sentencas for w in s]

uni = Counter(palavras)
N, V = sum(uni.values()), len(uni)

bi = Counter()
for s in sentencas:
    bi.update(zip([INICIO] + s, s))

contexto = Counter(uni)
contexto[INICIO] = len(sentencas)

LAMB = 0.7

def p_uni(w):
    return (uni[w] + 1) / (N + V)

def p_bi(prev, w):
    cond = bi[(prev, w)] / contexto[prev] if contexto[prev] else 0.0
    return LAMB * cond + (1 - LAMB) * p_uni(w)

def slor(frase):
    ws = re.findall(r"\w+", frase.lower())
    total_lm = total_uni = 0.0
    prev = INICIO
    print(frase)
    print(f"   {'token':<15}{'ln P(w|ant)':>12}{'ln P(w)':>10}{'contrib':>9}{'c(bi)':>7}")
    for w in ws:
        a, b = math.log(p_bi(prev, w)), math.log(p_uni(w))
        total_lm += a
        total_uni += b
        print(f"   {w:<15}{a:>12.2f}{b:>10.2f}{a - b:>9.2f}{bi[(prev, w)]:>7}")
        prev = w
    print(f"   SLOR = ({total_lm:.2f} - ({total_uni:.2f})) / {len(ws)} = {(total_lm - total_uni) / len(ws):.3f}\n")


print(f"corpus: {N:,} tokens | {V:,} formas | {len(sentencas):,} sentencas | " f"{len(bi):,} bigramas distintos")
print(f"ln(1 - λ) = ln({1 - LAMB:.1f}) = {math.log(1 - LAMB):.4f}\n")

slor("Há três dias o incêndio atingiu uma área de mata nativa")
slor("nativa uma incêndio há mata de o área três atingiu dias")

slor("Focos recorrentes apresentaram persistência anômala")
slor("As ideias verdes incolores dormem furiosamente")

slor("Os satélites registraram 47 focos")
slor("Os satélites registrou 47 focos")

corpus: 1,012,661 tokens | 53,115 formas | 51,115 sentencas | 359,873 bigramas distintos
ln(1 - λ) = ln(0.3) = -1.2040

Há três dias o incêndio atingiu uma área de mata nativa
   token           ln P(w|ant)   ln P(w)  contrib  c(bi)
   há                    -5.54     -6.72     1.18    259
   três                  -3.92     -7.10     3.18     36
   dias                  -3.73     -7.39     3.67     30
   o                     -3.23     -2.86    -0.37     21
   incêndio              -8.25    -10.41     2.16     22
   atingiu              -11.47    -10.27    -1.20      0
   uma                   -6.19     -4.98    -1.20      0
   área                  -6.37     -8.16     1.79     17
   de                    -1.09     -2.49     1.40    135
   mata                 -10.00     -9.99    -0.01      4
   nativa                -3.53    -11.58     8.04      2
   SLOR = (-63.32 - (-81.95)) / 11 = 1.694

nativa uma incêndio há mata de o área três atingiu dias
   token           ln P(w|ant)   ln P(w)

### Aplicação das fórmulas aos valores observados

$$\mathrm{SLOR}(s) = \frac{\ln P_{\mathrm{LM}}(s) - \ln P_{\mathrm{uni}}(s)}{|s|}$$

Os dois termos do numerador são somas sobre tokens, cada parcela proveniente de um
estimador distinto. A saída do código apresenta ambos já logaritmizados; explicita-se a
seguir o cálculo subjacente.

#### As constantes do corpus

```
N = 1.012.661        tokens
V =    53.115        formas distintas
N + V = 1.065.776    denominador do Laplace
λ = 0,7              ln(1 − λ) = −1,2040
```

#### Legenda das colunas

| coluna | é | fórmula |
|---|---|---|
| `w` | o token examinado | — |
| `c(w)` | ocorrências de $w$ no corpus | — |
| `P_uni` | probabilidade isolada, com Laplace | $\dfrac{c(w)+1}{N+V}$ |
| `lnP_uni` | logaritmo natural da anterior | $\ln P(w)$ |
| `c(a,w)` | ocorrências do par *(anterior, w)* | $c(w_{i-1}, w_i)$ |
| `c(a)` | ocorrências do token anterior — **o denominador** | $c(w_{i-1})$ |
| `MLE` | máxima verossimilhança do bigrama | $\dfrac{c(w_{i-1},w_i)}{c(w_{i-1})}$ |
| `L*MLE` | **primeiro** termo da interpolação | $\lambda \cdot \mathrm{MLE}$ |
| `(1-L)Puni` | **segundo** termo — o recuo para a unigrama | $(1-\lambda)\,P(w)$ |
| `P_bi` | a soma dos dois | $P(w_i \mid w_{i-1})$ |
| `lnP_bi` | logaritmo da anterior | $\ln P(w_i \mid w_{i-1})$ |
| `contrib` | a parcela que o token aporta ao numerador | $\ln P_{\mathrm{bi}} - \ln P_{\mathrm{uni}}$ |

Para o primeiro token da sentença, o antecessor é o marcador $\langle s \rangle$, com
$c(\langle s \rangle) = 51{.}115$ — o número de sentenças do corpus.

#### O estimador unigrama, avaliado

$$P(w) = \frac{c(w)+1}{N+V}$$

| $w$ | $c(w)$ | $P(w)$ | $\ln P(w)$ |
|---|---:|---:|---:|
| `de` | 88.320 | $8{,}29 \times 10^{-2}$ | $-2{,}49$ |
| `o` | 61.303 | $5{,}75 \times 10^{-2}$ | $-2{,}86$ |
| `uma` | 7.305 | $6{,}86 \times 10^{-3}$ | $-4{,}98$ |
| `há` | 1.281 | $1{,}20 \times 10^{-3}$ | $-6{,}72$ |
| `incêndio` | 31 | $3{,}00 \times 10^{-5}$ | $-10{,}41$ |
| `nativa` | 9 | $9{,}38 \times 10^{-6}$ | $-11{,}58$ |
| `anômala` | **0** | $9{,}38 \times 10^{-7}$ | $-13{,}88$ |

A última linha exemplifica o efeito da suavização de Laplace. A forma `anômala` não
ocorre no corpus; na ausência do $+1$ no numerador, ter-se-ia $P = 0$ e
$\ln P = -\infty$, tornando o SLOR indefinido. Com a suavização, a forma recebe o
**piso** $1/(N+V) = 9{,}38 \times 10^{-7}$, atribuído a qualquer forma não observada. O
piso não constitui escolha arbitrária: decorre da fórmula quando $c = 0$.

#### O estimador bigrama, com os dois termos separados

$$P(w_i \mid w_{i-1}) =
\underbrace{\lambda\,\frac{c(w_{i-1},w_i)}{c(w_{i-1})}}_{\text{termo condicional}}
\;+\;
\underbrace{(1-\lambda)\,P(w_i)}_{\text{recuo para a unigrama}}$$

Percorrendo a sentença de referência:

| $w_i$ | $c(a,w)$ | $c(a)$ | MLE | $\lambda\cdot$MLE | $(1{-}\lambda)P(w)$ | $P(w_i \mid w_{i-1})$ | contrib |
|---|---:|---:|---:|---:|---:|---:|---:|
| `há` | 259 | 51.115 | 0,00507 | 0,00355 | $3{,}61\times10^{-4}$ | $3{,}91\times10^{-3}$ | $+1{,}18$ |
| `três` | 36 | 1.281 | 0,02810 | 0,01967 | $2{,}47\times10^{-4}$ | $1{,}99\times10^{-2}$ | $+3{,}18$ |
| `dias` | 30 | 878 | 0,03417 | 0,02392 | $1{,}85\times10^{-4}$ | $2{,}41\times10^{-2}$ | $+3{,}67$ |
| `o` | 21 | 656 | 0,03201 | 0,02241 | $1{,}73\times10^{-2}$ | $3{,}97\times10^{-2}$ | $-0{,}37$ |
| `incêndio` | 22 | 61.303 | 0,00036 | 0,00025 | $9{,}01\times10^{-6}$ | $2{,}60\times10^{-4}$ | $+2{,}16$ |
| `atingiu` | **0** | 31 | 0 | **0** | $1{,}04\times10^{-5}$ | $1{,}04\times10^{-5}$ | $\mathbf{-1{,}20}$ |
| `uma` | **0** | 36 | 0 | **0** | $2{,}06\times10^{-3}$ | $2{,}06\times10^{-3}$ | $\mathbf{-1{,}20}$ |
| `área` | 17 | 7.305 | 0,00233 | 0,00163 | $8{,}59\times10^{-5}$ | $1{,}72\times10^{-3}$ | $+1{,}79$ |
| `de` | 135 | 304 | 0,44408 | 0,31086 | $2{,}49\times10^{-2}$ | $3{,}36\times10^{-1}$ | $+1{,}40$ |
| `mata` | 4 | 88.320 | 0,00005 | 0,00003 | $1{,}38\times10^{-5}$ | $4{,}55\times10^{-5}$ | $-0{,}01$ |
| `nativa` | 2 | 48 | 0,04167 | 0,02917 | $2{,}82\times10^{-6}$ | $2{,}92\times10^{-2}$ | $\mathbf{+8{,}04}$ |

A tabela autoriza três leituras.

**A maior contribuição da sentença provém de $c(a,w) = 2$.** O bigrama *mata nativa* foi
observado **duas vezes** em um milhão de tokens. O denominador, contudo, é reduzido —
`mata` ocorre 48 vezes —, de modo que $2/48 = 0{,}0417$ é valor elevado. Uma forma de
frequência muito baixa ($\ln P_{\mathrm{uni}} = -11{,}58$) eleva-se a $-3{,}53$ porque o
**contexto é igualmente raro**. Trata-se de evidência escassa produzindo efeito
considerável, e a fragilidade reside no denominador: uma ocorrência a menos reduziria a
contribuição à metade.

**A previsibilidade, isoladamente, não é suficiente: é necessário que exceda a
esperada.** A forma `de` apresenta MLE de $0{,}444$ — em quase metade das ocorrências de
`área`, `de` figura em seguida. A contribuição, entretanto, é de apenas $+1{,}40$, dado
que $P_{\mathrm{uni}}(\texttt{de})$ já era elevada e o segundo termo da interpolação,
$2{,}49 \times 10^{-2}$, dilui o ganho. É o efeito da subtração que define o SLOR.

**As duas linhas em negrito caracterizam a degeneração.** Quando $c(w_{i-1},w_i) = 0$, o
primeiro termo é **exatamente nulo**, e não meramente pequeno. Resta

$$P(w_i \mid w_{i-1}) = (1-\lambda)P(w_i)
\quad\Longrightarrow\quad
\text{contrib} = \ln\bigl[(1-\lambda)P(w_i)\bigr] - \ln P(w_i) = \ln(1-\lambda)$$

O fator $P(w_i)$ **cancela-se**. A contribuição independe da forma, de sua frequência e
do contexto. A medida não incorre em erro: deixa de mensurar contexto sem passar a
mensurar qualquer outra grandeza.

#### A sentença que degenera integralmente

| $w_i$ | $c(w_i)$ | $c(a,w)$ | $\lambda\cdot$MLE | contrib |
|---|---:|---:|---:|---:|
| `focos` | 4 | 0 | 0 | $-1{,}20$ |
| `recorrentes` | 2 | 0 | 0 | $-1{,}20$ |
| `apresentaram` | 30 | 0 | 0 | $-1{,}20$ |
| `persistência` | 4 | 0 | 0 | $-1{,}20$ |
| `anômala` | **0** | 0 | 0 | $-1{,}20$ |

Cinco tokens, cinco ocorrências de valor nulo no termo condicional. O primeiro termo da
interpolação **não é avaliado** em posição alguma, e

$$\mathrm{SLOR}(s) = \frac{5 \cdot \ln(1-\lambda)}{5} = \ln(1-\lambda) = -1{,}204$$

O valor **independe da sentença**. A coluna $c(w_i)$ é ilustrativa: as frequências variam
de 0 a 30, mais de uma ordem de grandeza, e a contribuição é idêntica nas cinco linhas —
demonstração aritmética de que o cancelamento é completo.

---

### Onde funciona

| frase | SLOR |
|---|---:|
| Há três dias o incêndio atingiu uma área de mata nativa | $+1{,}694$ |
| nativa uma incêndio há mata de o área três atingiu dias | $-1{,}000$ |

As **mesmas formas**, em idêntica quantidade. O termo $\ln P_{\mathrm{uni}}$ é idêntico
em ambas ($-81{,}95$), uma vez que a distribuição unigrama ignora a ordem. A totalidade
da diferença reside no primeiro termo — precisamente a grandeza que o SLOR foi
construído para isolar.

O sinal comporta-se conforme previsto: positivo na sentença bem ordenada, negativo na
embaralhada.

A coluna `contrib` explicita a origem do sinal, token a token:

```
nativa        +8,04   c(bi) =   2   "mata nativa" está atestado no corpus
dias          +3,67   c(bi) =  30   "três dias" é sequência frequente
três          +3,18   c(bi) =  36
incêndio      +2,16   c(bi) =  22
área          +1,79   c(bi) =  17
há            +1,18   c(bi) = 259   abertura de frase, medida pelo marcador <s>
```

O ganho não se distribui uniformemente: concentra-se nos poucos pares atestados pelo
corpus. É a estrutura que está sendo mensurada.

O maior deles merece exame. Isoladamente, `nativa` é forma de frequência muito baixa —
$\ln P = -11{,}58$ —, mas **na sequência de `mata`** o modelo praticamente a prevê, e a
surpresa decresce para $-3{,}53$. São oito nats de diferença derivados de um bigrama
observado **duas vezes** em um milhão de tokens: evidência escassa produzindo efeito
considerável, fragilidade que convém registrar.

A última linha é igualmente relevante: o marcador de início não constitui formalidade
notacional. Ele veicula a distribuição de **aberturas** — 259 sentenças do corpus
iniciam-se por `há` — e é o que diferencia as duas sentenças naquela posição.

Este é o resultado inacessível às técnicas anteriores. A sentença embaralhada apresenta
todas as formas no léxico (não detectada por T1), nenhum desvio de concordância efetivo
(não detectada por T2), e as regras acionadas caracterizaram outro fenômeno (T3 incorreu
em alvo incorreto). O defeito é **global**: não possui localização, não reside em token
algum. Apenas uma medida da sentença integral o detecta.

### Onde falha, e por quê

| frase | SLOR |
|---|---:|
| Focos recorrentes apresentaram persistência anômala | $-1{,}204$ |
| As ideias verdes incolores dormem furiosamente | $-0{,}939$ |

A primeira constitui português técnico correto e emprega três termos que nomeiam colunas
da própria base — `focos`, `recorrência`, `persistência`. A segunda é o exemplo clássico
de sentença gramaticalmente perfeita e semanticamente vazia. **Ambas recaem na mesma
faixa estreita, e a fórmula explicita a razão.**

A coluna `c(bi)` de ambas é reveladora: os bigramas são nulos em quase toda posição — na
primeira, **em todas**, inclusive a de abertura, dado que `focos` não inicia sentença
alguma no corpus. É a degeneração já derivada acima, agora sobre a sentença integral:
$\ln(1-\lambda) = -1{,}204$ exatos na primeira; $-0{,}939$ na segunda, porque apenas `as`
escapa, abrindo 858 sentenças do corpus.

Os $0{,}265$ que separam as duas sentenças decorrem, portanto, **inteiramente da forma
que abre cada uma**. Não intervêm gramática, sentido ou domínio.

**O mesmo colapso incide sobre o par mínimo, com resultado mais grave.** As duas últimas
sentenças da célula — *Os satélites registraram 47 focos* e *Os satélites registrou 47
focos* — recebem $-0{,}072$ **ambas**, dado que o bigrama do verbo não ocorre no corpus
em nenhuma das versões e o restante é idêntico. Um desvio de concordância verbal
permanece indetectado.

Trata-se do mesmo desvio que a técnica 2 localiza por token e traço e que o LanguageTool
não detecta: três técnicas, três respostas distintas para a mesma sentença.

### Implicações

O SLOR não incorreu em erro: **não dispunha de objeto a mensurar**. Um corpus de um
milhão de tokens contém 360 mil bigramas distintos, ao passo que o espaço possível é
$V^2 \approx 2{,}8$ bilhões. A cobertura é de $0{,}013\%$.

O problema é agudo neste domínio. O mac_morpho consiste em texto jornalístico de 1994:
registra `incêndio`, `área` e `mata`, mas não `foco` na acepção de foco de calor,
tampouco nomes de satélite, `frp` ou `persistência` na acepção adotada pela base. **A
sentença de referência obteve $+1{,}694$ unicamente por ter sido redigida com o
vocabulário que o corpus fortuitamente cobre.** A substituição de `mata nativa` por
`vegetação remanescente` reduziria o escore sem alteração alguma na qualidade do texto.

Este é o problema da **esparsidade**, e a razão de existirem os modelos neurais: eles
generalizam para sequências não observadas, atribuindo probabilidade a partir de
similaridade distribucional em lugar de contagem direta. É precisamente essa lacuna que
as técnicas subsequentes se propõem a suprir.

### Quando usar

- **Entre sentenças de conteúdos distintos.** É a finalidade da subtração: descontar a
  frequência isolada permite comparar sentenças que não compartilham vocabulário —
  terreno em que o PLL falha.
- **Para defeito global, sem localização.** A sentença embaralhada apresenta todas as
  formas no léxico, concordância íntegra e nenhuma regra acionada. Apenas uma medida da
  sentença integral a detecta.
- **Interpretando o sinal, e não somente a ordenação.** Valor positivo indica que a
  ordem acrescenta probabilidade; negativo, que a sequência é anti-estrutural. É uma das
  poucas medidas examinadas cujo zero possui significado.

### Quando não usar

- **Com estimador esparso.** Quando os bigramas da sentença não ocorrem no corpus, a
  medida degenera: cada token contribui $\ln(1-\lambda)$ e o resultado converte-se em
  constante. O sintoma é observável na saída — a coluna `c(bi)` integralmente nula.
- **Como detector de desvio local.** Devolve um valor por sentença, sem indicar token
  algum.
- **Na presença de forma fora do léxico.** A correção por frequência pressupõe contagens
  confiáveis; forma desconhecida recai no piso e compromete o resultado.
- **Confiando no valor sem examinar a evidência subjacente.** No par mínimo o SLOR
  atribuiu $-0{,}072$ a **ambas** as sentenças — valor idêntico para a correta e para a
  agramatical, sem qualquer indicação do empate na saída. A coluna de zeros constituía o
  indício de que não havia objeto a mensurar.


<a id="t05-pll"></a>

## PLL — Pseudo-Log-Likelihood (pseudo-log-verossimilhança)

O SLOR estima a probabilidade sob um modelo **causal**, restrito ao contexto à esquerda.
Um modelo **mascarado** dispõe de ambos os lados: suprime-se um token e consulta-se o
modelo sobre o valor esperado naquela posição, com a totalidade da sentença remanescente
disponível.

A pseudo-log-verossimilhança soma essas respostas, uma por posição:

$$\boxed{\;\mathrm{PLL}(s) = \sum_{i=1}^{|s|} \ln P_{\mathrm{MLM}}\bigl(s_i \mid s_{\setminus i}\bigr)\;}$$

em que $s_{\setminus i}$ designa a sentença com a posição $i$ substituída pela marca
`[MASK]`. Cada parcela exige **uma passagem** pelo modelo: uma sentença de $n$ tokens
demanda $n$ inferências.

### Por que "pseudo"

O qualificativo é literal, e não ressalva notacional. As $|s|$ parcelas se sobrepõem:
cada uma pressupõe **corretas todas as demais posições**. A soma, portanto, não
corresponde a $\ln P(s)$ — não há regra de cadeia que a produza, e ela não integra 1
sobre o espaço das sentenças.

Decorre daí que **o valor absoluto carece de significado**; apenas comparações são
interpretáveis.

### Soma ou média

A soma é grandeza **extensiva**: cada posição acrescenta uma parcela negativa, de modo
que o escore decresce com o comprimento ainda que a sentença seja bem formada. A
comparação de sentenças de extensões distintas pela soma mensura sobretudo a extensão.

A média é **intensiva** — escore por posição, comparável entre extensões:

$$\overline{\mathrm{PLL}}(s) = \frac{1}{|s|}\sum_{i=1}^{|s|} \ln P_{\mathrm{MLM}}\bigl(s_i \mid s_{\setminus i}\bigr)$$

O custo é o descarte da informação de que uma sentença longa acumula mais evidência.
Adota-se a média, que ademais a torna homogênea ao SLOR, igualmente dividido por $|s|$.

### A diferença que determina o emprego

Confrontando-se com o SLOR:

$$\mathrm{SLOR}(s) = \frac{\ln P_{\mathrm{LM}}(s) - \overbrace{\ln P_{\mathrm{uni}}(s)}^{\text{referência}}}{|s|}
\qquad\qquad
\overline{\mathrm{PLL}}(s) = \frac{\ln P_{\mathrm{MLM}}(s) - \overbrace{0}^{\text{nenhuma}}}{|s|}$$

**O PLL não subtrai referência alguma.** Trata-se da mesma forma com a linha de base
anulada, e essa ausência constitui simultaneamente sua vantagem e sua limitação.


In [6]:
# --- só para PLL e verbo mascarado ---
%conda install -c conda-forge -y pytorch-cpu transformers


Channels:
 - conda-forge
Platform: win-64
Solving environment: ...working... done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODELO = "neuralmind/bert-base-portuguese-cased"
tok = AutoTokenizer.from_pretrained(MODELO)
mlm = AutoModelForMaskedLM.from_pretrained(MODELO)
mlm.eval()


@torch.no_grad()
def pll(frase, detalhe=False):
    ids = tok(frase, return_tensors="pt")["input_ids"][0]
    soma, pecas = 0.0, []
    for i in range(1, len(ids) - 1):              # pula [CLS] e [SEP]
        masc = ids.clone()
        orig = masc[i].item()
        masc[i] = tok.mask_token_id               # apaga a posição i
        lp = torch.log_softmax(mlm(masc.unsqueeze(0)).logits[0, i], -1)[orig].item()
        soma += lp
        pecas.append((tok.convert_ids_to_tokens([orig])[0], lp))
    print(frase)
    if detalhe:
        for p, lp in pecas:
            print(f"   {p:<16}{lp:>8.2f}{'  <<<' if lp < -5 else ''}")
    print(f"   |s| = {len(pecas)}   soma = {soma:.2f}   média = {soma / len(pecas):.3f}\n")


pll("Há três dias o incêndio atingiu uma área de mata nativa.", detalhe=True)
pll("Há três dias os incêndios atingiu uma área de mata nativa.", detalhe=True)

pll("Há três dias o incêndio cantou uma área de mata nativa.")
pll("nativa uma incêndio há mata de o área três atingiu dias.")


c:\Users\Matheus\miniforge3\envs\pln\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 203/203 [00:00<00:00, 30343.68it/s]
[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Há três dias o incêndio atingiu uma área de mata nativa.
   Há                 -0.43
   três               -2.05
   dias               -0.59
   o                  -2.14
   incêndio           -1.95
   atingiu            -0.91
   uma                -0.06
   área               -0.07
   de                 -0.04
   mata               -0.21
   nativa             -0.70
   .                  -0.00
   |s| = 12   soma = -9.16   média = -0.763

Há três dias os incêndios atingiu uma área de mata nativa.
   Há                 -1.41
   três               -2.33
   dias               -0.90
   os                 -1.47
   incêndios          -1.65
   atingiu            -6.09  <<<
   uma                -0.22
   área               -0.09
   de                 -0.04
   mata               -0.26
   nativa             -0.69
   .                  -0.00
   |s| = 12   soma = -15.16   média = -1.263

Há três dias o incêndio cantou uma área de mata nativa.
   |s| = 12   soma = -26.30   média = -2.192

nativa uma inc

### Aplicação das fórmulas aos valores observados

$$\mathrm{PLL}(s) = \sum_{i=1}^{|s|} \ln P_{\mathrm{MLM}}\bigl(s_i \mid s_{\setminus i}\bigr)$$

O objeto central da fórmula é $s_{\setminus i}$, a sentença com a posição $i$ suprimida.
É construído $|s|$ vezes, uma por posição, e não figura na saída do código; explicita-se
a seguir.

#### O que a soma percorre

A soma percorre **peças do vocabulário**, e não formas ortográficas. A sentença de
referência é submetida ao modelo na seguinte segmentação:

```
['[CLS]', 'Há', 'três', 'dias', 'o', 'incêndio', 'atingiu',
 'uma', 'área', 'de', 'mata', 'nativa', '.', '[SEP]']
```

`[CLS]` e `[SEP]` permanecem fora do laço, restando $|s| = 12$. Neste caso, cada forma
foi preservada íntegra pelo vocabulário do BERTimbau — circunstância particular do
exemplo, e não regra geral: formas fora do vocabulário são segmentadas em subpalavras, e
a soma passa a comportar mais parcelas do que formas há na sentença. Nas quatro sentenças
desta seção o valor é 12 em todas, o que torna indiferente a escolha entre **soma** e
**média** neste conjunto, dado que a divisão de doze valores por doze não altera a
ordenação. A escolha torna-se relevante apenas quando as extensões divergem.

#### Duas parcelas, explicitadas por inteiro

Tomando a sentença com desvio de concordância verbal e abrindo as duas posições em que o
desvio se manifesta:

$i = 5$, suprimindo o substantivo:

```
s_sem_5 = Há três dias os [MASK] atingiu uma área de mata nativa .
```

| candidato | $\ln P$ |
|---|---:|
| **`incêndios`** ← o escrito | $\mathbf{-1{,}65}$ |
| `raios` | $-2{,}41$ |
| `ataques` | $-2{,}79$ |
| `animais` | $-3{,}03$ |

$i = 6$, suprimindo o verbo:

```
s_sem_6 = Há três dias os incêndios [MASK] uma área de mata nativa .
```

| candidato | $\ln P$ |
|---|---:|
| `atingiram` | $-0{,}22$ |
| `atingem` | $-2{,}35$ |
| `invadiram` | $-3{,}26$ |
| ⋮ | |
| **`atingiu`** ← o escrito | $\mathbf{-6{,}09}$ |

**A forma no plural é a mais provável na posição 5.** O modelo dispõe de `os ... atingiu`,
com verbo no singular à direita, e ainda assim atribui a `incêndios` a maior
probabilidade. A parcela $i=5$ não penaliza o desvio: corrobora-o.

#### As doze parcelas, lado a lado

| $i$ | $s_i$ | $\ln P$ (certa) | $\ln P$ (errada) | $\Delta$ |
|---:|---|---:|---:|---:|
| 1 | `Há` | $-0{,}43$ | $-1{,}41$ | $-0{,}98$ |
| 2 | `três` | $-2{,}05$ | $-2{,}33$ | $-0{,}28$ |
| 3 | `dias` | $-0{,}59$ | $-0{,}90$ | $-0{,}31$ |
| 4 | `o` / `os` | $-2{,}14$ | $-1{,}47$ | $\mathbf{+0{,}67}$ |
| 5 | `incêndio` / `incêndios` | $-1{,}95$ | $-1{,}65$ | $\mathbf{+0{,}30}$ |
| 6 | `atingiu` | $-0{,}91$ | $-6{,}09$ | $\mathbf{-5{,}18}$ |
| 7 | `uma` | $-0{,}06$ | $-0{,}22$ | $-0{,}16$ |
| 8 | `área` | $-0{,}07$ | $-0{,}09$ | $-0{,}02$ |
| 9 | `de` | $-0{,}04$ | $-0{,}04$ | $0{,}00$ |
| 10 | `mata` | $-0{,}21$ | $-0{,}26$ | $-0{,}05$ |
| 11 | `nativa` | $-0{,}70$ | $-0{,}69$ | $+0{,}01$ |
| 12 | `.` | $-0{,}00$ | $-0{,}00$ | $0{,}00$ |
| | **soma** | $\mathbf{-9{,}16}$ | $\mathbf{-15{,}16}$ | $\mathbf{-6{,}00}$ |

A coluna $\Delta$ merece exame detido, por registrar um comportamento contraintuitivo.

**As duas posições em que o desvio foi introduzido apresentaram melhora.** A alteração
`o` → `os` produziu ganho de $0{,}67$; `incêndio` → `incêndios`, de $0{,}30$. E
`atingiu` — **a única forma idêntica nas duas sentenças** — decresceu $5{,}18$,
correspondendo a 86% da queda total. A penalização recai sobre a posição não alterada.

#### A explicação, na fórmula

O desvio de concordância constitui **relação** entre as posições 4–5 e a posição 6. A
fórmula, por sua vez, é

$$\mathrm{PLL}(s) = \sum_i \ln P(s_i \mid s_{\setminus i})$$

isto é, uma soma de termos **indexados por posição**. Não existe termo
$\Phi(s_4, s_5, s_6)$, nem qualquer termo que tome duas posições simultaneamente. Um
desvio de natureza relacional deve, portanto, ser imputado a alguma posição, e a
atribuição é determinada pelo condicionamento: em $i=5$ o modelo dispõe de `os` à
esquerda e toma o plural como contexto estabelecido; em $i=6$ toma `os incêndios` como
dado e conclui pela inadequação do verbo.

Cada parcela pressupõe **correto todo o restante**. As parcelas 5 e 6 formulam
pressuposições mutuamente contraditórias — uma assume o plural, a outra o singular — e a
soma não dispõe de meio para detectá-lo. É precisamente o que o prefixo *pseudo*
designa: não se trata de $\ln P(s)$, mas da soma de doze condicionais mutuamente
incompatíveis.

**A consequência para fins de auditoria é imediata.** O PLL indica a posição 6, mas a
sentença admite correção igualmente adequada em 6 (`os incêndios atingiram`) ou em 4–5
(`o incêndio atingiu`). A medida não distingue desvio no verbo de desvio no sujeito:
enuncia apenas que a posição 6 é improvável dado o restante. A localização devolvida é
artefato de qual constituinte o modelo tomou como fixo, e não a posição efetiva do
defeito.

#### A inexistência de piso

Cabe examinar, por fim, a sentença **correta**:

```
s_sem_5 = Há três dias o [MASK] atingiu uma área de mata nativa .
   fogo      -0,70        <- vence
   incêndio  -1,95        <- o escrito
```

```
s_sem_6 = Há três dias o incêndio [MASK] uma área de mata nativa .
   destruiu  -0,72        <- vence
   atingiu   -0,91        <- o escrito
```

Em uma sentença isenta de defeito, a forma escrita é **superada** em ambas as posições:
`fogo` é mais provável que `incêndio`, e `destruiu` mais provável que `atingiu`. Não se
trata de falha do modelo: $\ln P$ mensura tipicidade, e a forma mais típica raramente
coincide com a escolhida pelo autor.

Daí decorre a inexistência de limiar. A forma `incêndio` registra $-1{,}95$ estando
correta; `atingiu` registra $-6{,}09$ estando incorreta; `três` registra $-2{,}05$ e `o`
registra $-2{,}14$, ambas corretas e ambas inferiores a `incêndios` incorreta, a
$-1{,}65$. Não há ponto de corte em $\ln P$ que separe escolha incomum de desvio, dado
que a fórmula não computa grandezas distintas nos dois casos: computa a mesma grandeza,
baixa por ambas as razões.

---

### Onde funciona

Entre as duas versões do par mínimo, a separação é nítida — $-0{,}763$ contra
$-1{,}263$. Mais instrutiva, contudo, é a
**localização** da diferença: as duas sentenças são quase idênticas da metade em diante
— `área` registra −0,07 contra −0,09, `de` registra −0,04 em ambas, `nativa` −0,70
contra −0,69 —, e a queda concentra-se em um único token, `atingiu`, com **−6,09** contra
os −0,91 da versão correta.

Manifesta-se aqui a bidirecionalidade, de forma mais rica que em um par comum. Ao
mascarar o verbo, o modelo dispõe de `os incêndios` **à esquerda** e conclui pela
adequação do plural — inferência acessível também a um modelo causal. As formas `Há`,
`três` e `dias`, contudo, **decresceram** na sentença com desvio, e são anteriores ao
sujeito. A penalização só é possível porque o modelo, ao avaliá-las, já dispunha de
`atingiu` **à direita**, em conflito com o plural intercalado. A informação flui em ambos
os sentidos.

Cabe registrar o que **não** decresceu: `os` e `incêndios` obtiveram valores superiores
aos de `o` e `incêndio` (−1,47 e −1,65 contra −2,14 e −1,95). Isoladamente, o plural
constitui abertura mais provável; o modelo não penaliza o sintagma, mas a **relação**
deste com o verbo.

Trata-se precisamente do par em que o SLOR falhou: ali, ambas as versões receberam valor
idêntico, uma vez que nenhum dos bigramas do verbo ocorria no corpus e a fórmula
degenerou. O modelo mascarado dispensa a observação prévia da sequência: generaliza.

### Onde falha, e por quê

| frase | $\overline{\mathrm{PLL}}$ | |
|---|---:|---|
| Há três dias o incêndio **cantou** uma área de mata nativa. | $-2{,}192$ | **anômala** |
| nativa uma incêndio há mata de o área três atingiu dias. | $-10{,}414$ | **embaralhada** |

O PLL apresenta desempenho satisfatório neste caso, e é justamente por isso que o exemplo
é enganoso. As duas sentenças compartilham o vocabulário da base, o que torna a
comparação legítima: no interior do par mínimo, o viés de frequência se cancela.

O problema emerge fora desse contexto. **O PLL não subtrai referência.** O escore agrega
duas grandezas que não distingue — o grau de boa formação da sentença e o grau de
previsibilidade de suas formas. A substituição de `mata nativa` por `vegetação
remanescente`, sem alteração gramatical, reduz o escore: o modelo estará mensurando
raridade de vocabulário e apresentando o resultado como se fosse gramaticalidade.

No interior do par mínimo isso não ocorre, porque as duas sentenças **compartilham quase
todo o vocabulário** — o viés é idêntico em ambos os lados e se cancela na comparação.
Fora dele, nada opera esse cancelamento.

### A regra de emprego, e a simetria com o SLOR

> **O PLL é válido no interior do par mínimo. O SLOR é válido entre sentenças de
> conteúdos distintos.** O emprego de um no domínio do outro constitui o equívoco
> recorrente.

As duas técnicas falham em direções **opostas**, e pela mesma razão estrutural:

| | referência subtraída | condição de falha |
|---|---|---|
| SLOR | frequência unigrama | não há evidência no corpus (esparsidade) |
| PLL | nenhuma | o conteúdo varia (pune vocabulário menos previsível) |

O SLOR dispõe da correção de frequência e por isso atravessa conteúdos distintos, mas
depende de contagens que um corpus reduzido não fornece, e o vocabulário deste domínio
encontra-se em grande parte ausente dele. O PLL generaliza para qualquer sequência, mas
sem referência não distingue *raro* de *incorreto*.

Nenhum dos dois localiza o defeito: ambos devolvem um valor para a sentença integral. O
detalhamento apresentado acima constitui instrumento de análise adotado neste documento;
a técnica, conforme definida, entrega apenas a média.

### Quando usar

- **No interior do par mínimo.** É o domínio próprio da medida: as duas sentenças
  compartilham quase todo o vocabulário, o viés de frequência é idêntico em ambos os
  lados e se cancela na comparação.
- **Quando se exige generalização do estimador.** Foi precisamente onde o SLOR degenerou:
  os bigramas `incêndio atingiu` e `incêndios atingiu` não ocorriam no corpus e a fórmula
  devolveu o piso. O modelo neural dispensa a observação prévia da sequência.
- **Sob a forma de média, quando as extensões divergirem.** A soma é extensiva e decresce
  com o comprimento ainda que a sentença seja bem formada; comparar sentenças de
  extensões distintas pela soma mensura sobretudo o comprimento.

### Quando não usar

- **Entre sentenças de conteúdos distintos.** Na ausência de referência subtraída, o
  escore penaliza vocabulário menos previsível, e a sentença correta de vocabulário
  técnico é superada pela sentença trivial.
- **Como valor absoluto.** As parcelas se sobrepõem e a soma não integra 1 sobre o espaço
  das sentenças. Não constitui probabilidade; apenas comparações são interpretáveis.
- **Para localizar o defeito.** A técnica entrega um valor por sentença; o detalhamento
  posição a posição não integra a definição.
- **Comparando valores entre modelos ou tokenizadores distintos.** O escore depende da
  segmentação da sentença em peças; a substituição do modelo altera todos os valores.


<a id="t06-raiz"></a>

## Teste do núcleo predicativo (teste da raiz)

Toda sentença requer um predicado. Não se trata de heurística nem de tendência
estatística, mas de propriedade sintática. Em uma árvore de dependências, essa
propriedade é verificável em **um único ponto**: a raiz, que por construção é o nó do
qual todos os demais dependem.

### A formulação

Seja $\rho(s)$ a raiz da árvore de $s$ e $\mathrm{filhos}(\rho)$ seus dependentes
diretos. A sentença possui predicado quando

$$\mathrm{predicado}(s) \iff
\underbrace{\mathrm{pos}(\rho) \in \{\texttt{VERB}, \texttt{AUX}\}}_{\text{predicado verbal}}
\;\;\lor\;\;
\underbrace{\exists\, c \in \mathrm{filhos}(\rho) : \mathrm{dep}(c) = \texttt{cop}}_{\text{predicado nominal}}$$

Duas linhas de código e um único nó inspecionado: é a técnica de menor custo entre as
examinadas que ainda assim produz veredito binário **e** localizado.

### Por que a raiz, e não a presença de verbo

O teste ingênuo — a verificação da existência de algum verbo na sentença — aprova o
fragmento. Confrontem-se:

- *O incêndio **atingiu** a mata nativa.*
- *O incêndio que **atingiu** a mata nativa.*

Ambas contêm o verbo `atingiu`; a primeira constitui oração, a segunda não. O que difere
não é a presença do verbo, mas sua **posição** na árvore: o pronome `que` o rebaixa a
oração relativa (`acl:relcl`) e promove o substantivo à raiz.

Um único token converte oração em fragmento, e **apenas a estrutura registra o fato**.
Contagem de verbos, busca de padrões e mensuração de probabilidade não alcançam essa
distinção.

### Por que a cláusula da cópula

Em predicações nominais — *O carro é azul* — a raiz é o predicativo `azul`, etiquetado
`ADJ`. O verbo `é` figura como dependente, com função `cop`. Na ausência do segundo termo
da disjunção, **toda construção copular seria classificada como fragmento**, o que
inviabilizaria a técnica em português.

### O escopo da medida

A técnica mensura **completude**, e nada além disso. Não determina se a sentença é
correta, se é semanticamente coerente ou se é bem redigida: determina apenas se possui
predicado. O escopo é explícito, e essa delimitação é o que a torna confiável naquilo a
que se propõe.


In [8]:
import spacy

nlp = spacy.load("pt_core_news_md")


def nucleo(frase, arvore=False):
    doc = nlp(frase)
    raiz = [t for t in doc if t.dep_ == "ROOT"][0]
    copula = any(f.dep_ == "cop" for f in raiz.children)
    tem_predicado = raiz.pos_ in ("VERB", "AUX") or copula

    print(frase)
    if arvore:
        for t in doc:
            seta = "  <== RAIZ" if t.dep_ == "ROOT" else ""
            print(f"   {t.text:<12}{t.pos_:<7}{t.dep_:<10}-> {t.head.text}{seta}")
    forma = raiz.morph.get("VerbForm")            # só informativo: o teste não olha
    print(f"   raiz = {raiz.text} ({raiz.pos_}"
          f"{', VerbForm=' + forma[0] if forma else ''})"
          f"{'  + cópula' if copula else ''}")
    print(f"   veredito: {'ORAÇÃO' if tem_predicado else 'FRAGMENTO'}\n")


nucleo("Há três dias o incêndio atingiu uma área de mata nativa.", arvore=True)
nucleo("Há três dias o incêndio que atingiu uma área de mata nativa.", arvore=True)
nucleo("O incêndio é intenso.", arvore=True)

nucleo("Atingir uma área de mata nativa em três dias.", arvore=True)


Há três dias o incêndio atingiu uma área de mata nativa.
   Há          VERB   case      -> dias
   três        NUM    nummod    -> dias
   dias        NOUN   obl       -> atingiu
   o           DET    det       -> incêndio
   incêndio    NOUN   nsubj     -> atingiu
   atingiu     VERB   ROOT      -> atingiu  <== RAIZ
   uma         DET    det       -> área
   área        NOUN   obj       -> atingiu
   de          ADP    case      -> mata
   mata        NOUN   nmod      -> área
   nativa      ADJ    amod      -> área
   .           PUNCT  punct     -> atingiu
   raiz = atingiu (VERB, VerbForm=Fin)
   veredito: ORAÇÃO

Há três dias o incêndio que atingiu uma área de mata nativa.
   Há          VERB   case      -> dias
   três        NUM    nummod    -> dias
   dias        NOUN   ROOT      -> dias  <== RAIZ
   o           DET    det       -> incêndio
   incêndio    NOUN   ROOT      -> incêndio  <== RAIZ
   que         PRON   nsubj     -> atingiu
   atingiu     VERB   acl:relcl -> incêndi

### Aplicação das fórmulas aos valores observados

A árvore de dependências é o par $T(s) = (V, A)$, com $V$ o conjunto de tokens e $A$ o de
arcos rotulados. A raiz é o vértice que aponta para si mesmo:

$$r = v \in V \ \text{ tal que } \ (v, \texttt{ROOT}, v) \in A$$

O teste é uma disjunção de duas condições sobre esse único vértice:

$$\mathrm{Pred}(s) \;=\; \underbrace{\bigl[\mathrm{pos}(r) \in \{\text{VERB}, \text{AUX}\}\bigr]}_{D_1}
\;\lor\;
\underbrace{\bigl[\exists c \in \mathrm{filhos}(r) : \mathrm{dep}(c) = \texttt{cop}\bigr]}_{D_2}$$

$$\mathrm{Veredito}(s) = \begin{cases} \text{ORAÇÃO} & \text{se } \mathrm{Pred}(s) \\
\text{FRAGMENTO} & \text{caso contrário}\end{cases}$$

Cabe observar o que a fórmula **não** contém: nenhuma referência ao conjunto de verbos da
sentença. Ela inspeciona um único vértice, e convém examinar a razão disso.

#### A raiz não é escolhida — é lida do conjunto de arcos

Apresentam-se abaixo os arcos das duas sentenças, com a única diferença destacada:

```
base        Há    case       -> dias          fragmento   Há    case       -> dias
            três  nummod     -> dias                      três  nummod     -> dias
            dias  obl        -> atingiu                   dias  ROOT       -> dias      <<<
            o     det        -> incêndio                  o     det        -> incêndio
            incên nsubj      -> atingiu                   incên ROOT       -> incêndio  <<<
                                                          que   nsubj      -> atingiu
            ating ROOT       -> atingiu       <<<         ating acl:relcl  -> incêndio  <<<
            uma   det        -> área                      uma   det        -> área
            área  obj        -> atingiu                   área  obj        -> atingiu
            de    case       -> mata                      de    case       -> mata
            mata  nmod       -> área                      mata  nmod       -> área
            nativa amod      -> área                      nativa amod      -> área
```

O ponto central da técnica reside na comparação dos conjuntos de **verbos** das duas
sentenças:

$$\{t \in V : \mathrm{pos}(t) \in \{\text{VERB},\text{AUX}\}\} \;=\; \{\texttt{Há},\ \texttt{atingiu}\}
\qquad \text{em ambas.}$$

**Conjuntos idênticos, vereditos opostos.** As duas sentenças apresentam os mesmos dois
verbos, na mesma ordem e com as mesmas etiquetas morfológicas. A introdução da forma `que`
alterou o rótulo de um arco — `atingiu` deixou de ser `ROOT` e passou a `acl:relcl` de
`incêndio` —, e isso é suficiente para inverter o veredito. Constitui demonstração de que
o teste opera sobre $A$, e não sobre $V$: a presença de verbo é invariante entre as duas,
e a medida ainda assim as distingue.

Cabe registrar que `Há` recebe a etiqueta `VERB` mas ocupa o arco `case`, com função
preposicional. Um teste baseado na mera existência de verbo identificaria dois verbos no
fragmento e o aprovaria.

#### As duas condições, avaliadas

| frase | $r$ | $\mathrm{pos}(r)$ | $D_1$ | filho `cop` | $D_2$ | $D_1 \lor D_2$ |
|---|---|---|:---:|---|:---:|---|
| base | `atingiu` | VERB | **V** | — | F | ORAÇÃO |
| fragmento | `dias` | NOUN | F | — | F | **FRAGMENTO** |
| cópula | `intenso` | ADJ | F | `é` | **V** | ORAÇÃO |
| infinitivo | `Atingir` | VERB | **V** | — | F | ORAÇÃO |
| anomalia | `cantou` | VERB | **V** | — | F | ORAÇÃO |
| ordem embaralhada | `atingiu` | VERB | **V** | — | F | ORAÇÃO |

A terceira linha justifica a segunda cláusula. Em *O incêndio é intenso*, a raiz é
`intenso`, um **adjetivo**, de modo que $D_1$ é falso. O verbo `é` está presente, porém
subordinado como `cop` **abaixo** da raiz. Na ausência de $D_2$, uma oração perfeitamente
formada seria rejeitada: o padrão Universal Dependencies estabelece o predicativo como
núcleo e a cópula como dependente, e a fórmula deve acompanhar essa convenção.

#### Uma pressuposição que o analisador não garante

A definição estabelece *o* vértice tal que $(v,\texttt{ROOT},v) \in A$. No fragmento:

$$R = \{v : (v,\texttt{ROOT},v) \in A\} = \{\texttt{dias},\ \texttt{incêndio}\}
\qquad |R| = 2$$

$T(s)$ **não constitui árvore**, mas floresta de dois componentes. A raiz única inexiste,
e o artigo definido da fórmula permanece sem referente. A implementação resolve o caso
com `[0]`, tomando o primeiro vértice na ordem linear da sentença.

No caso presente o veredito é preservado fortuitamente: `dias` e `incêndio` são ambos
`NOUN`, de modo que $D_1 \lor D_2$ é falso para ambos e a resposta é FRAGMENTO em
qualquer hipótese. Caso um dos componentes apresentasse núcleo verbal e o outro nominal,
o resultado dependeria da ordem de ocorrência. Formalmente configuram-se **três testes
distintos**, e a fórmula não especifica qual deles é o pertinente:

$$\forall r \in R:\ \mathrm{Pred}(r)
\qquad\qquad
\exists r \in R:\ \mathrm{Pred}(r)
\qquad\qquad
\mathrm{Pred}(r_{\text{primeiro}})$$

A implementação adota o terceiro, único dos três desprovido de justificativa linguística.

#### Três casos aprovados indevidamente

**`Atingir uma área de mata nativa em três dias.`** — raiz `Atingir`, `pos = VERB`, $D_1$
verdadeiro, veredito ORAÇÃO. Trata-se, contudo, de fragmento: infinitivo isolado não
predica. A implementação chega a computar e imprimir a forma verbal, com o comentário
`# só informativo: o teste não olha`. O reforço de $D_1$ com a exigência de
`VerbForm=Fin` capturaria este caso, ao custo seguinte:

```
Foram detectados 47 focos na região norte.
   r = detectados   pos = VERB   VerbForm = Part
   D1 com exigência de Fin  ->  FALSO      (oração legítima recusada)
```

Na voz passiva o particípio ocupa a raiz e o auxiliar finito é **filho** dela. A exigência
de `Fin` sobre $r$ rejeita toda construção passiva. A correção demanda uma terceira
cláusula — $D_3$: existência de filho `aux`/`aux:pass`/`cop` com `VerbForm=Fin` —, e cada
cláusula adicional corresponde a uma convenção do padrão de anotação que o teste passa a
incorporar.

O mesmo se aplica ao gerúndio (*Avançando pela encosta na madrugada*, `VerbForm=Ger`) e
às orações subordinadas isoladas (*Depois de atingir a mata nativa*, `Inf`). Três
construções, um único modo de falha.

**`Há três dias o incêndio cantou uma área de mata nativa.`** — raiz `cantou`, VERB,
ORAÇÃO. O resultado é correto: a estrutura *é* completa. A predicação de *cantar* sobre
*incêndio* constitui problema de outra natureza, e $\mathrm{Pred}$ não comporta termo
algum que examine o conteúdo semântico dos vértices.

**`nativa uma incêndio há mata de o área três atingiu dias.`** — raiz `atingiu`, VERB,
ORAÇÃO. Este é o caso mais problemático. A sentença é ininteligível e o analisador
produziu árvore anômala (`mata` recebeu a etiqueta `VERB`, `dias` foi analisado como
objeto de `atingiu`); ainda assim, identificou-se núcleo verbal e o teste aprovou a
sentença.

A razão está na fórmula: a imagem de $\mathrm{Pred}$ é $\{0,1\}$ — um único bit, que
registra a **existência** de núcleo predicativo, existência essa verificada. A expressão
não comporta grandeza alguma que avalie a adequação dos demais arcos; tal avaliação exige
outra medida, com outro domínio.

---

### A correção, e seu custo

O ajuste mínimo consiste em exigir finitude — insuficiente para a passiva, como
demonstrado acima, mas suficiente para explicitar a natureza da falha:

$$\mathrm{predicado}(s) \iff
\bigl[\mathrm{pos}(\rho) \in \{\texttt{VERB}, \texttt{AUX}\} \;\land\;
\mathrm{VerbForm}(\rho) = \texttt{Fin}\bigr]
\;\lor\; \exists\, c : \mathrm{dep}(c) = \texttt{cop}$$

Esse ajuste é revelador quando confrontado com as técnicas anteriores. A falha
não decorreu de insuficiência de informação nem de limitação do modelo, mas de que **a
definição transcrita estava incompleta**. A proposição *toda sentença possui predicado* é
verdadeira; a proposição *todo predicado é um verbo na raiz* não o é — o predicado deve
ser um verbo **finito**.

Trata-se de modalidade de falha distinta das anteriores. Na técnica 2 o insumo era
incorreto; na 4, faltava evidência; na 5, faltava referência. Aqui todos os componentes
operaram conforme o projeto, e o projeto é que era incompleto.

### Quando usar

- **Invariavelmente, e em etapa inicial.** Duas linhas de código, um único nó
  inspecionado, veredito binário **e** localizado. Apresenta a melhor relação entre custo
  e exatidão entre as técnicas examinadas.
- **Para detecção de fragmento**, defeito da *forma* da árvore: não há token defeituoso a
  localizar, e nenhuma contagem de verbos ou medida de probabilidade alcança a distinção.
- **Previamente às técnicas que pressupõem predicado.** O verbo mascarado devolve n/a
  quando a raiz não é verbo; este teste fornece a mesma informação antecipadamente e a
  custo muito inferior.

### Quando não usar

- **Sem exigência de finitude.** Na formulação atual, a fórmula testa a **classe** do nó
  raiz, e não sua **flexão**: *Correr todos os dias pela manhã* é aprovado porque
  `Correr` é `VERB`. Gerúndio e subordinada isolada falham do mesmo modo, e o dado que
  resolveria o caso (`VerbForm`) já consta da saída.
- **Como medida de qualidade.** Responde a uma única questão — a existência de predicado
  — e nada estabelece quanto a correção, sentido ou clareza da sentença.
- **Sem verificação da árvore.** O veredito depende integralmente do nó eleito raiz pelo
  analisador. Um erro de análise inverte a resposta sem qualquer indicação na saída.


<a id="t07-complexidade"></a>

## Índices de complexidade sintática

As técnicas anteriores investigavam a correção da sentença. Esta investiga outra
propriedade: **o grau de elaboração da construção**. Não avalia correção; descreve a
forma de construção, e o resultado que produz não é um veredito binário, mas um perfil
numérico.

Três índices se extraem da mesma árvore.

### Profundidade máxima

A distância do nó mais afastado até a raiz:

$$\mathrm{prof}(s) = \max_{t \in s} \; d(t, \rho)$$

em que $d(t,\rho)$ designa o número de arcos percorridos de $t$ até a raiz. O índice
mensura **encaixamento**: o número de níveis de subordinação acumulados pela sentença.

### Número de orações

Contabilizam-se os nós que encabeçam oração — a principal e as subordinadas:

$$\mathrm{orac}(s) = \bigl|\{\, t \in s : \mathrm{dep}(t) \in \mathcal{C} \,\}\bigr|,
\qquad \mathcal{C} = \{\texttt{ROOT}, \texttt{ccomp}, \texttt{xcomp}, \texttt{advcl},
\texttt{acl}, \texttt{acl:relcl}, \texttt{csubj}\}$$

### Ramificação média

O número médio de filhos por nó:

$$\mathrm{ramif}(s) = \frac{1}{n}\sum_{t \in s} \bigl|\mathrm{filhos}(t)\bigr|$$

O índice aparenta mensurar a extensão horizontal da árvore. **Convém examinar essa
terceira fórmula antes de adotá-la**; o exemplo adiante explicita a razão.

### Finalidade

Comparação de registros (jornalístico e jurídico), estimativa de dificuldade de leitura,
caracterização autoral. São medidas de **estilo e carga de processamento**, e não de
correção. Uma sentença de elevada complexidade não é por isso incorreta, e uma sentença
incorreta pode ser trivialmente simples.


In [9]:
import spacy

nlp = spacy.load("pt_core_news_md")

CLAUSAIS = {"ROOT", "ccomp", "xcomp", "advcl", "acl", "acl:relcl", "csubj"}


def profundidade(t):
    d = 0
    while t.dep_ != "ROOT":
        t, d = t.head, d + 1
    return d


def complexidade(frase):
    doc = nlp(frase)
    n = len(doc)
    prof = max(profundidade(t) for t in doc)
    oracoes = sum(1 for t in doc if t.dep_ in CLAUSAIS)
    arcos = sum(len(list(t.children)) for t in doc)
    print(f"   {n:>3}{prof:>7}{oracoes:>9}{arcos / n:>13.4f}{(n - 1) / n:>10.4f}   {frase}")


print(f"   {'n':>3}{'prof.':>7}{'orações':>9}{'ramificação':>13}{'(n-1)/n':>10}   frase")
complexidade("O incêndio devastou mata, cerrado, pasto, encosta e vale.")
complexidade("O incêndio que atingiu a mata que cercava o vale enfim cessou.")


     n  prof.  orações  ramificação   (n-1)/n   frase
    13      3        1       0.9231    0.9231   O incêndio devastou mata, cerrado, pasto, encosta e vale.
    13      6        3       0.9231    0.9231   O incêndio que atingiu a mata que cercava o vale enfim cessou.


As duas sentenças apresentam **exatamente 13 tokens** e estruturas maximamente
distintas: a primeira constitui lista plana — um verbo e cinco irmãos dele dependentes;
a segunda encaixa duas relativas encadeadas, uma no interior da outra.

### Aplicação das fórmulas aos valores observados

Os três índices operam sobre a mesma árvore, cada qual percorrendo-a de modo distinto.
Percorrem-se a seguir os três sobre a sentença encaixada.

#### $\mathrm{prof}$ — um máximo sobre caminhos

$$\mathrm{prof}(s) = \max_{t \in s} \; d(t, \rho)$$

$d(t,\rho)$ é o comprimento do caminho de $t$ até a raiz, caminho que existe para todo
token. É esse conjunto de caminhos que a fórmula percorre, um a um, antes de tomar o
máximo:

| $t$ | caminho $t \to \rho$ | $d(t,\rho)$ |
|---|---|---:|
| `cessou` | `cessou` | 0 |
| `incêndio` | `incêndio → cessou` | 1 |
| `enfim` | `enfim → cessou` | 1 |
| `O` | `O → incêndio → cessou` | 2 |
| `atingiu` | `atingiu → incêndio → cessou` | 2 |
| `que`₁ | `que → atingiu → incêndio → cessou` | 3 |
| `mata` | `mata → atingiu → incêndio → cessou` | 3 |
| `a` | `a → mata → atingiu → incêndio → cessou` | 4 |
| `cercava` | `cercava → mata → atingiu → incêndio → cessou` | 4 |
| `que`₂ | `que → cercava → mata → atingiu → incêndio → cessou` | 5 |
| `vale` | `vale → cercava → mata → atingiu → incêndio → cessou` | 5 |
| **`o`** | **`o → vale → cercava → mata → atingiu → incêndio → cessou`** | **6** |

$$\mathrm{prof}(s) = \max\{0,1,1,2,2,3,3,4,4,5,5,6\} = 6$$

Cabe observar **qual token** realiza o máximo: `o`, um artigo. O índice registra seis
níveis de encaixamento, e o token responsável por esse valor é uma forma funcional em
posição terminal. O máximo é sempre atingido em uma folha, e folhas são tipicamente
determinantes e preposições. O percurso informativo é o intermediário (`atingiu`,
`cercava`), precisamente o que a fórmula descarta ao reter apenas o extremo.

Na lista plana o mesmo percurso resulta em $\max = 3$, e as folhas de maior profundidade
são as vírgulas.

#### $\mathrm{orac}$ — a cardinalidade de um filtro

$$\mathrm{orac}(s) = \bigl|\{\, t \in s : \mathrm{dep}(t) \in \mathcal{C} \,\}\bigr|$$

O filtro aplicado, token a token:

| frase | tokens com $\mathrm{dep} \in \mathcal{C}$ | $\mathrm{orac}$ |
|---|---|---:|
| lista plana | `devastou` (ROOT) | 1 |
| encaixada | `cessou` (ROOT), `atingiu` (acl:relcl), `cercava` (acl:relcl) | 3 |

Na lista plana, `cerrado`, `pasto`, `encosta` e `vale` recebem `conj`, rótulo que **não
pertence** a $\mathcal{C}$. O tratamento é adequado: trata-se de substantivos
coordenados, e não de orações.

A mesma etiqueta, contudo, ocorre na coordenação de **verbos**:

```
O incêndio devastou a mata e destruiu o vale.
   devastou   VERB   ROOT   -> devastou     [C]
   destruiu   VERB   conj   -> devastou           <- fora de C
   orac = 1
```

Duas orações, contagem 1. A lacuna é legível diretamente na definição de $\mathcal{C}$:
o conjunto enumera rótulos de **subordinação**, e coordenação não é subordinação. A
contabilização de coordenadas exigiria admitir `conj`, o que faria as quatro coordenadas
nominais da lista plana serem igualmente computadas como orações. Não há escolha de
$\mathcal{C}$ adequada a ambos os casos, uma vez que o rótulo não distingue a categoria
do elemento coordenado.

#### $\mathrm{ramif}$ — a soma que não varia

$$\mathrm{ramif}(s) = \frac{1}{n}\sum_{t \in s} \bigl|\mathrm{filhos}(t)\bigr|$$

O numerador, mensurado nas três sentenças:

| frase | $n$ | $\sum_t \lvert\mathrm{filhos}(t)\rvert$ | $\mathrm{ramif}$ | $(n-1)/n$ |
|---|---:|---:|---:|---:|
| lista plana | 13 | **12** | 0,9231 | 0,9231 |
| encaixada | 13 | **12** | 0,9231 | 0,9231 |
| `Há três dias o incêndio atingiu…` | 12 | **11** | 0,9167 | 0,9167 |

A soma equivale a $n-1$ em todas as três. A subseção seguinte explicita a razão.

### Onde funciona

**A profundidade e o número de orações separam as duas com margem ampla.** A lista plana
alcança profundidade 3 e uma única oração; a sentença encaixada alcança profundidade 6 e
três orações. O encaixamento duplicou e a contagem de orações triplicou, sem alteração do
comprimento.

É precisamente o comportamento esperado de um índice de complexidade: capturar a **forma**
da árvore independentemente do número de formas que a compõem. Nenhuma das técnicas
anteriores distingue essas duas sentenças, uma vez que ambas são corretas.

### Onde falha, e por quê

**A ramificação média é idêntica nas duas — 0,9231 — e coincide com $(n-1)/n$ até a
quarta casa decimal.** Não se trata de coincidência, mas de identidade.

Em uma árvore, **todo nó exceto a raiz possui exatamente um pai**. O número de arcos é,
portanto, fixo:

$$\sum_{t \in s} \bigl|\mathrm{filhos}(t)\bigr| = n - 1$$

e, consequentemente,

$$\mathrm{ramif}(s) = \frac{n-1}{n} = 1 - \frac{1}{n}$$

**A ramificação média não depende da árvore.** Constitui função exclusiva do comprimento
— monótona, crescente e limitada por 1. Duas sentenças de mesmo $n$ recebem sempre o
mesmo valor, seja a árvore uma lista plana ou uma escada de subordinadas.

(A identidade pressupõe **raiz única**. Quando o analisador devolve uma floresta — como
no fragmento da seção 6, com duas raízes —, a soma decresce para $n-2$ e o índice vale
$(n-2)/n$. Continua não mensurando a forma da árvore: passa a mensurar o número de
componentes.)

O índice apresenta a aparência de medida estrutural: soma filhos, divide por nós e
produz valor decimal plausível entre 0 e 1. Não veicula, contudo, **nenhuma** informação
estrutural. A obtenção da ramificação média requer apenas a contagem de formas; a árvore
é dispensável.

### Implicações metodológicas

Uma métrica pode ser bem definida, computável e estável e ainda assim não mensurar
grandeza alguma. Neste caso a falha não reside no dado nem no modelo, mas na omissão da
verificação algébrica prévia.

O teste que a detecta é de baixo custo e recomendável como procedimento padrão:
**verificar a existência de identidade**. Antes da adoção de qualquer índice, cabe
determinar se ele não constitui função determinística de grandeza já disponível —
comprimento, contagem de tokens, número de nós. Em caso afirmativo, o índice não
acrescenta dimensão alguma, limitando-se a reapresentar um dado preexistente sob outra
denominação.

A variante potencialmente informativa está próxima: a divisão pelo número de nós **não
terminais** faz o denominador variar com a forma da árvore. Ela de fato separa as duas:

| frase | $n-1$ | não terminais | razão |
|---|---:|---:|---:|
| lista plana | 12 | 7 | 1,714 |
| encaixada | 12 | 6 | 2,000 |
| `Há três dias o incêndio atingiu…` | 11 | 5 | **2,200** |

A terceira linha, contudo, invalida a proposta. A sentença de referência — plana, de
profundidade 3 e uma única oração — recebe o **maior** valor das três, superior ao da
escada de relativas. A variante distingue as árvores, mas não as **ordena** por
complexidade: mensura o número médio de filhos por nó ramificador, de modo que uma
sentença curta e larga supera uma sentença profunda e estreita. A correção do denominador
eliminou a identidade sem produzir a medida pretendida, o que corrobora o procedimento
proposto acima em vez de dispensá-lo.

### Quando usar

- **Para caracterização de registro e estimativa de carga de leitura** — comparação entre
  o jurídico e o jornalístico, descrição autoral, estimativa de dificuldade. São medidas
  de **estilo**, e é essa a sua finalidade.
- **A profundidade e o número de orações**, que efetivamente capturam a forma da árvore
  independentemente do comprimento: as duas sentenças do exemplo possuem 13 tokens cada e
  distinguem-se por 3 contra 6 e por 1 contra 3.
- **Como caracterização, em conjunto com outras medidas**, nunca como veredito isolado.

### Quando não usar

- **Como detector de desvio.** As duas sentenças do exemplo são corretas. Sentença
  complexa não é por isso incorreta, e sentença incorreta pode ser trivialmente simples:
  não é essa a questão a que os índices respondem.
- **A ramificação média, em hipótese alguma.** Equivale a $(n-1)/n$ por identidade:
  constitui função exclusiva do comprimento e não veicula informação alguma sobre a
  árvore. Sua obtenção requer apenas a contagem de formas.
- **Sem verificação algébrica prévia.** É o procedimento que esta seção propõe: diante de
  qualquer índice novo, determinar se ele não constitui função determinística de grandeza
  já disponível. Em caso afirmativo, o índice reapresenta um dado sob outra denominação.


<a id="t08-dependencia"></a>

## Comprimento de dependência e não-projetividade

As técnicas anteriores examinavam **quais** formas se vinculam a quais. Esta examina **a
distância** que as separa na ordem linear e a eventual sobreposição entre os vínculos.

São duas medidas extraídas da mesma árvore, ambas operando sobre apenas duas informações:
o conjunto de arcos e a posição de cada forma. Não intervêm etiquetas, morfologia ou
léxico.

### Comprimento de dependência

Cada forma $t$, exceto a raiz, vincula-se ao seu núcleo $h(t)$. Sendo $i(\cdot)$ a posição
na sentença, o arco apresenta comprimento

$$d(t) = \bigl|\, i(t) - i(h(t)) \,\bigr|$$

Somando-se sobre a sentença integral:

$$D(s) = \sum_{t \,\neq\, \rho} d(t)
\qquad\qquad
\bar{D}(s) = \frac{D(s)}{|A|}$$

em que $|A|$ designa o número de arcos; a pontuação é excluída do cômputo. As duas
grandezas mensuram propriedades distintas: $D$ cresce com a extensão da sentença, de modo que apenas $\bar{D}$ é
comparável entre sentenças de comprimentos diferentes.

**Fundamentação da medida.** Um arco vincula duas formas cuja relação é necessária à
interpretação da sentença. Ao processar a primeira, o leitor mantém o vínculo em aberto
até encontrar a segunda; quanto maior o arco, maior o intervalo de retenção. É a hipótese
da **minimização do comprimento de dependência**: as línguas tendem a ordenar as formas
de modo a manter $D$ reduzido.

### Não-projetividade

Tomem-se dois arcos, cada um representado com a extremidade de menor índice em primeiro
lugar:
$(i,j)$ e $(k,l)$, com $i<j$ e $k<l$. Os arcos **se cruzam** quando exatamente uma extremidade
de cada um recai no interior do outro:

$$i < k < j < l \qquad\text{ou}\qquad k < i < l < j$$

O total de pares que se cruzam é

$$X(s) = \bigl|\,\{\, \{a,b\} \subseteq A \;:\; a \text{ cruza } b \,\}\,\bigr|$$

e a árvore é **projetiva** quando $X(s) = 0$, isto é, quando é possível traçar todos os
arcos acima da linha sem sobreposição entre eles.

```
        PROJETIVO                        NÃO-PROJETIVO

    ┌───────────────┐                ┌───────────┐
    │     ┌─────┐   │                │     ┌─────┼─────┐
    A     B     C   D                A     B     C     D

    (1,4) contém (2,3):              (1,3) e (2,4):
    um dentro do outro               uma extremidade de cada dentro do outro
```

A condição é puramente aritmética sobre índices. Não comporta elemento linguístico algum:
trata-se de geometria de intervalos.


In [10]:
import spacy
from itertools import combinations

nlp = spacy.load("pt_core_news_md")


def arcos(doc):
    return [(t.i, t.head.i) for t in doc
            if t.head.i != t.i and t.pos_ != "PUNCT"]


def cruzam(a, b):
    i, j = sorted(a)
    k, l = sorted(b)
    return i < k < j < l or k < i < l < j


def dependencia(frase, detalhe=False):
    doc = nlp(frase)
    A = arcos(doc)
    dist = [abs(d - h) for d, h in A]

    print(frase)
    if detalhe:
        for t in doc:
            if t.head.i != t.i and t.pos_ != "PUNCT":
                print(f"   {t.text:<11}-> {t.head.text:<11}{abs(t.i - t.head.i):>3}   [{t.dep_}]")
    print(f"   |A| = {len(A)}   D = {sum(dist)}   D/|A| = {sum(dist)/len(A):.2f}   "
          f"maior arco = {max(dist)}   "
          f"cruzamentos = {sum(1 for x, y in combinations(A, 2) if cruzam(x, y))}\n")


dependencia("O vento agravou a seca que intensificou o fogo que destruiu a mata.", detalhe=True)
dependencia("O fogo que a seca que o vento agravou intensificou destruiu a mata.", detalhe=True)


O vento agravou a seca que intensificou o fogo que destruiu a mata.
   O          -> vento        1   [det]
   vento      -> agravou      1   [nsubj]
   a          -> seca         1   [det]
   seca       -> agravou      2   [obj]
   que        -> intensificou  1   [nsubj]
   intensificou-> seca         2   [acl:relcl]
   o          -> fogo         1   [det]
   fogo       -> intensificou  2   [obj]
   que        -> destruiu     1   [nsubj]
   destruiu   -> fogo         2   [acl:relcl]
   a          -> mata         1   [det]
   mata       -> destruiu     2   [obj]
   |A| = 12   D = 17   D/|A| = 1.42   maior arco = 2   cruzamentos = 0

O fogo que a seca que o vento agravou intensificou destruiu a mata.
   O          -> fogo         1   [det]
   fogo       -> destruiu     9   [nsubj]
   que        -> intensificou  7   [obj]
   a          -> seca         1   [det]
   seca       -> intensificou  5   [nsubj]
   que        -> agravou      3   [obj]
   o          -> vento        1   [det]
   ve

### Aplicação das fórmulas aos valores observados

As duas medidas operam sobre apenas duas informações: o conjunto de arcos e as posições.
Percorrem-se a seguir sobre a sentença de referência.

#### $d(t)$ e $D(s)$ sobre a sentença de referência

```
[0]Há  [1]três  [2]dias  [3]o  [4]incêndio  [5]atingiu  [6]uma  [7]área  [8]de  [9]mata  [10]nativa
```

$$d(t) = \bigl|\, i(t) - i(h(t)) \,\bigr|$$

| $t$ | $i(t)$ | $h(t)$ | $i(h(t))$ | $d(t)$ |
|---|---:|---|---:|---:|
| `Há` | 0 | `dias` | 2 | 2 |
| `três` | 1 | `dias` | 2 | 1 |
| `dias` | 2 | `atingiu` | 5 | 3 |
| `o` | 3 | `incêndio` | 4 | 1 |
| `incêndio` | 4 | `atingiu` | 5 | 1 |
| `uma` | 6 | `área` | 7 | 1 |
| `área` | 7 | `atingiu` | 5 | 2 |
| `de` | 8 | `mata` | 9 | 1 |
| `mata` | 9 | `área` | 7 | 2 |
| `nativa` | 10 | `área` | 7 | 3 |

$$D(s) = 2+1+3+1+1+1+2+1+2+3 = 17
\qquad
\bar{D}(s) = \frac{17}{10} = 1{,}70$$

#### O predicado de cruzamento, avaliado em pares reais

$$a \text{ cruza } b \iff i < k < j < l \ \text{ ou } \ k < i < l < j
\qquad (i<j,\ k<l)$$

São $\binom{10}{2} = 45$ pares. Explicitam-se quatro deles, com as desigualdades
correspondentes:

| $a$ | $b$ | teste | resultado |
|---|---|---|---|
| `dias→atingiu` (2,5) | `o→incêndio` (3,4) | $2<3<5<4$? **não** ($5 \not< 4$) | encaixado |
| `área→atingiu` (5,7) | `mata→área` (7,9) | $5<7<7$? **não** ($7 \not< 7$) | extremidade comum |
| `Há→dias` (0,2) | `área→atingiu` (5,7) | $0<5<2$? **não** | disjuntos |
| `mata→área` (7,9) | `nativa→área` (7,10) | $7<7$? **não** | extremidade comum |

Os 45 pares recaem integralmente em uma dessas três situações, e $X(s) = 0$. Cabe notar
que as desigualdades são **estritas**: extremidades compartilhadas não configuram
cruzamento,
propriedade que torna uma árvore-estrela — com todos os nós dependentes de um único
vértice — invariavelmente projetiva.

#### O comportamento de $\bar{D}$ sobre a sentença embaralhada

| frase | $\lvert A\rvert$ | $D$ | $\bar{D}$ | maior arco | $X$ |
|---|---:|---:|---:|---:|---:|
| `Há três dias o incêndio atingiu uma área de mata nativa.` | 10 | 17 | **1,70** | 3 | 0 |
| `nativa uma incêndio há mata de o área três atingiu dias.` | 10 | 20 | **2,00** | 7 | 0 |

Este é o resultado problemático da seção. A segunda sentença é ininteligível — as onze
formas foram permutadas aleatoriamente —, e $\bar{D}$ eleva-se de 1,70 para **2,00**,
permanecendo no interior da faixa que a própria seção estabelece como típica da prosa em
português.

O maior arco é mais informativo ($3 \to 7$, dado que `incêndio` passou a ocupar posição
sete unidades anterior à do verbo), mas $\bar{D}$ é uma **média**, e a média dilui: nove
arcos curtos absorvem um arco de comprimento sete. Não há limiar em $\bar{D}$ que separe
sequência aleatória de prosa comum, uma vez que a sequência aleatória, submetida à
análise, produz árvore de arcos majoritariamente curtos como qualquer outra.

#### Por que $X$ permanece nulo

A seção sustenta que $X = 0$ constitui o valor esperado, e não informação. A afirmação é
verificável. Doze sentenças foram submetidas à fórmula — a de referência, a embaralhada,
o fragmento, extraposições, uma interrogativa com deslocamento, um aposto entre vírgulas
e três construções elaboradas deliberadamente para produzir cruzamento:

$$X(s) = 0 \quad \text{em todas as doze.}$$

O mecanismo torna-se visível ao se examinar a árvore produzida pelo analisador:

```
Uma área foi atingida de mata nativa pelo incêndio.
   [1] área      nsubj:pass -> [3] atingida
   [5] mata      obj        -> [3] atingida      <- deveria pender de "área"
   [8] incêndio  obl:agent  -> [3] atingida
```

O sintagma `de mata nativa` modifica **área**, e não `atingida`: a análise correta
vincularia `mata` a `área`, produzindo o arco $(1,5)$. Com esse arco, somado ao arco
$(3,8)$ do agente:

$$1 < 3 < 5 < 8 \quad\Longrightarrow\quad \text{cruzam},\qquad X(s) = 1$$

A árvore **correta é não-projetiva**; a devolvida pelo analisador não o é, e a diferença
reside em um único arco: o modificador deslocado foi vinculado à raiz em lugar do
substantivo. Em uma árvore cujos nós dependem todos da raiz não há cruzamento algum — a
raiz não possui arco de entrada, e os demais arcos permanecem encaixados.

Em síntese: $X$ não mensura a sentença, mas o analisador. Este foi treinado sobre banco de
árvores majoritariamente projetivo e reproduz essa propriedade inclusive onde ela não se
aplica. O valor nulo exibido na coluna é característica da ferramenta, e não fato relativo
ao texto.


---

### Lendo a saída

| frase | $D$ | $\bar{D}$ | maior arco | $X$ |
|---|---:|---:|---:|---:|
| O vento agravou a seca que intensificou o fogo que destruiu a mata. | 17 | 1,42 | 2 | 0 |
| O fogo que a seca que o vento agravou intensificou destruiu a mata. | 43 | 3,58 | 9 | 0 |

As duas sentenças apresentam **as mesmas 13 formas e os mesmos 12 arcos**, e ambas são
corretas. Difere apenas a ordem — e $\bar{D}$ mais que duplica.

Na primeira, **nenhum arco excede 2**: cada relativa vincula-se ao substantivo
imediatamente anterior, e a sentença se resolve da esquerda para a direita sem retenção.
Na segunda, os vínculos se acumulam: `fogo` encontra seu verbo `destruiu` nove posições
adiante, e `intensificou` encontra seu núcleo oito posições atrás. O leitor deve manter
três sujeitos em aberto antes da ocorrência do primeiro verbo.

**A comparação deve recair sobre $\bar{D}$, e não sobre $D$.** O total cresce com o
número de arcos, de modo que, entre sentenças de extensões diferentes, $D$ mensura
sobretudo comprimento. No par em exame ambas apresentam $|A| = 12$, o que torna o total
igualmente comparável — propriedade deste par, e não da medida.

Como referência aproximada: a prosa comum em português situa-se em $\bar{D}$ entre 1 e 2.
Acima de 3, a sentença mantém número elevado de vínculos simultaneamente em aberto.

Este é o aspecto inacessível à seção anterior. Os índices de complexidade atribuiriam a
estas duas sentenças a mesma profundidade e o mesmo número de orações, dado que ambas
encaixam duas relativas. O que as distingue é a **posição** em que as relativas foram
inseridas, propriedade mensurada apenas pelo comprimento de dependência.

E $X = 0$ em ambas, como ocorre em quase todos os casos — o que conduz ao limite da
segunda medida.

### Quando usar

- **Para comparação entre arranjos do mesmo conteúdo.** É o emprego mais robusto: mesmas
  formas, mesmos vínculos, ordens distintas. A medida indica qual arranjo impõe maior
  custo de processamento.
- **Para caracterização de registro.** Texto jurídico e texto jornalístico diferem
  sistematicamente em $\bar{D}$; a medida presta-se à descrição de um gênero ou de um
  autor.
- **Para estimativa de dificuldade de leitura**, em conjunto com outras medidas, nunca
  isoladamente.

### Quando não usar

- **Como detector de desvio.** As duas sentenças do exemplo são corretas. Valor elevado
  de $\bar{D}$ indica custo de processamento, e não incorreção; uma sentença incorreta
  pode apresentar $\bar{D}$ muito baixo. Não é essa a questão a que a medida responde.
- **Com $D$ bruto entre sentenças de extensões diferentes.** Nesse caso mensura-se a
  extensão da sentença por via indireta.
- **Para não-projetividade em sentença isolada de português escrito.** $X$ é praticamente
  sempre nulo. O português escrito é majoritariamente projetivo e, nos poucos casos em
  que não o seria — uma relativa deslocada para posição final, por exemplo —, o
  analisador tende a devolver árvore projetiva de qualquer modo, vinculando a oração ao
  substantivo mais próximo. Um valor $X = 0$ não constitui, portanto, informação: é o
  valor esperado. A medida só é produtiva sobre corpus extenso ou em línguas de ordem
  mais livre, como o tcheco e o alemão.
- **Sem considerar a procedência dos arcos.** $D$ e $X$ são computados sobre a árvore
  **prevista**, e não sobre a árvore correta. Um arco vinculado à posição errada altera
  ambos os valores sem qualquer indicação na saída. A medida vale o que valer a análise
  sintática subjacente.


<a id="pipeline"></a>

## A pipeline — as técnicas em cadeia

As oito seções anteriores mensuraram cada técnica isoladamente. Esta as encadeia — e a
**ordem não constitui escolha de projeto, mas consequência do que foi mensurado.**

Três das quatro etapas derivam de técnicas já apresentadas. A quarta, o verbo mascarado,
é exposta adiante, na seção 11; figura aqui por constituir a única verificação semântica
empregada pela pipeline, e a pipeline é o ponto em que as etapas se encadeiam.

### Por que esta ordem, e não outra

A posição de cada etapa decorre de uma medição que a determina.

**A ortografia ocupa a primeira posição** por constituir o único veredito exato entre as
técnicas examinadas e porque uma forma fora do léxico não permanece contida nessa etapa:
deforma a árvore e, com ela, as duas etapas que a leem. Quando essa etapa produz achado,
as seguintes **prosseguem, porém com marcação** — a marca não indica incorreção, mas
menor confiabilidade.

**A estrutura precede a semântica** porque o verbo mascarado devolve `n/a` quando a raiz
não é verbo, conforme detalhado na seção 11. A existência de predicado é pré-requisito
para avaliar a adequação do verbo, e o teste da raiz a verifica em milissegundos.

### O que ficou de fora, e por quê

As medições constam das seções indicadas; duas delas são posteriores a esta.

| técnica | medição que a excluiu |
|---|---|
| Surpresa por token (seção 10) | o pico caiu no verbo **correto** da sentença de referência |
| SLOR (seção 4) | atribuiu valor idêntico ao par mínimo; degenera fora do vocabulário do corpus |
| Embeddings (seção 12) | antônimos mais próximos que sinônimos; a média apaga a ordem |

Restaram **o léxico, duas técnicas do spaCy e uma do BERTimbau**, acrescidas de três
correções identificadas nas seções anteriores: a exigência de verbo finito na raiz, a
comparação de gênero no particípio passivo e o tratamento do numeral por valor, dado que
este não porta traço de número.

### O que ela não cobre

Duas limitações, que convém enunciar previamente à execução.

**A escolha lexical.** *A três dias* atravessa todas as etapas sem detecção: `a` consta
do léxico, é preposição e não estabelece arco de concordância, e a etapa semântica não
inspeciona essa posição. Era o único caso detectado pelo LanguageTool.

**O erro factual.** A sentença *A área cresceu de 1284 para 340 hectares* é português
correto e é incorreta apenas em relação à base de dados. Nenhuma técnica de natureza
linguística alcança esse nível: a verificação exige o confronto do texto com os campos
que o originaram.


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

tok_b = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
mlm_b = AutoModelForMaskedLM.from_pretrained("neuralmind/bert-base-portuguese-cased")
mlm_b.eval()

TERMOS_DO_DOMINIO = set()      # especialização: {"fusionado", "frp", ...}
MARGEM_SUSPEITA = 5.0          # nats; acima disso o verbo destoa do esperado

NOMINAL = {"det", "amod", "acl"}
VERBAL = {"nsubj", "nsubj:pass"}
UM = {"1", "um", "uma"}


# --- 1. ortografia: nome próprio e número não contam como erro -------------
def etapa_ortografia(doc):
    proprios = {t.text.lower() for t in doc if t.pos_ == "PROPN"}
    return [(p, lexico.correction(p))
            for p in lexico.unknown(lexico.split_words(doc.text))
            if not p.isdigit() and p not in proprios and p not in TERMOS_DO_DOMINIO]


# --- 2. estrutura: exige verbo FINITO na raiz, ou auxiliar/cópula finito ---
def etapa_estrutura(doc):
    raiz = [t for t in doc if t.dep_ == "ROOT"][0]
    finito = lambda t: t.pos_ in ("VERB", "AUX") and t.morph.get("VerbForm") == ["Fin"]
    apoio = [c for c in raiz.children if c.dep_ in ("aux", "aux:pass", "cop")]
    if finito(raiz) or any(finito(c) for c in apoio):
        return None
    forma = raiz.morph.get("VerbForm")
    return f"sem predicado: raiz '{raiz.text}' é {raiz.pos_}" + (f"/{forma[0]}" if forma else "")


# --- 3. gramática: igualdade de traços entre nós ligados -------------------
def etapa_gramatica(doc):
    erros = []
    for t in doc:
        h = t.head
        tracos = ()
        if t.dep_ in NOMINAL and h.pos_ in ("NOUN", "PROPN"):
            tracos = ("Number", "Gender")
        elif t.dep_ in VERBAL and h.pos_ in ("VERB", "AUX"):
            tracos = ("Number", "Person")
            if h.morph.get("VerbForm") == ["Part"]:          # particípio passivo
                tracos += ("Gender",)
        for traco in tracos:
            a, b = t.morph.get(traco), h.morph.get(traco)
            if a and b and a != b:
                erros.append(f"{traco}: '{t.text}'({a[0]}) x '{h.text}'({b[0]}) [{t.dep_}]")

        if t.dep_ == "nummod" and h.pos_ == "NOUN":          # numeral não tem Number
            esperado = "Sing" if t.text.lower() in UM else "Plur"
            n = h.morph.get("Number")
            if n and n[0] != esperado:
                erros.append(f"Number: '{t.text}' pede {esperado}, '{h.text}' é {n[0]} [nummod]")
    return erros


# --- 4. semântica: margem do verbo da raiz --------------------------------
@torch.no_grad()
def etapa_semantica(doc, k=5):
    raiz = [t for t in doc if t.dep_ == "ROOT"][0]
    if raiz.pos_ != "VERB":
        return None, "n/a — a raiz não é verbo"
    ids = tok_b(raiz.text, add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        return None, f"n/a — '{raiz.text}' não é peça única do vocabulário"
    mascarada = "".join((tok_b.mask_token if t.i == raiz.i else t.text) + t.whitespace_
                        for t in doc)
    enc = tok_b(mascarada, return_tensors="pt")
    pos = int((enc["input_ids"][0] == tok_b.mask_token_id).nonzero()[0])
    lp = torch.log_softmax(mlm_b(**enc).logits[0, pos], dim=-1)
    topo = torch.topk(lp, k)
    margem = float(topo.values[0]) - float(lp[ids[0]])
    esperados = [tok_b.convert_ids_to_tokens([int(i)])[0] for i in topo.indices]
    return margem, f"'{raiz.text}' margem {margem:.2f}; esperados: {esperados}"


# --- o laudo: todas as etapas rodam sempre -------------------------------
def avaliar(frase):
    doc = nlp(frase)
    laudo = []

    orto = etapa_ortografia(doc)
    laudo.append(("ortografia", not orto,
                  ", ".join(f"{p} -> {c}" for p, c in orto) or "—"))

    # palavra fora do léxico compromete a análise sintática — e com ela as duas
    # etapas que leem a árvore. Elas continuam rodando, mas ficam marcadas.
    suspeita = f"   [árvore sob suspeita: {', '.join(p for p, _ in orto)}]" if orto else ""

    frag = etapa_estrutura(doc)
    laudo.append(("estrutura", frag is None, (frag or "predicado presente") + suspeita))

    gram = etapa_gramatica(doc)
    laudo.append(("gramática", not gram, ("; ".join(gram) or "—") + suspeita))

    margem, detalhe = etapa_semantica(doc)      # devolve n/a sozinha se não se aplicar
    laudo.append(("semântica", None if margem is None else margem < MARGEM_SUSPEITA, detalhe))

    return laudo


def relatorio(texto):
    """Etapa 0: segmenta. Depois roda as quatro etapas em cada frase."""
    for i, s in enumerate(nlp(texto).sents, 1):
        frase = s.text.strip()
        if not frase:
            continue
        print(f"\n[{i}] {frase}")
        for etapa, ok, detalhe in avaliar(frase):
            marca = "n/a" if ok is None else ("ok " if ok else "!! ")
            print(f"     {marca} {etapa:<15}{detalhe}")


relatorio("Há três dias o incêndio atingiu uma áreas de mata nativa. "
          "Há tres dias o incêndio atingiu uma áreas de mata natvia.")



Loading weights: 100%|██████████| 203/203 [00:00<00:00, 33684.52it/s]
[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[1] Há três dias o incêndio atingiu uma áreas de mata nativa.
     ok  ortografia     —
     ok  estrutura      predicado presente
     !!  gramática      Number: 'uma'(Sing) x 'áreas'(Plur) [det]
     ok  semântica      'atingiu' margem 0.65; esperados: ['destruiu', 'atingiu', 'invadiu', 'atinge', 'em']

[2] Há tres dias o incêndio atingiu uma áreas de mata natvia.
     !!  ortografia     natvia -> nativa
     ok  estrutura      predicado presente   [árvore sob suspeita: natvia]
     !!  gramática      Number: 'uma'(Sing) x 'áreas'(Plur) [det]   [árvore sob suspeita: natvia]
     ok  semântica      'atingiu' margem 0.95; esperados: ['destruiu', 'atingiu', 'invadiu', 'atinge', 'em']


### Aplicação das fórmulas aos valores observados

Toma-se a sentença

```
Há tres dias o incêndio atingiu uma áreas de mata natvia.
```

e percorrem-se as quatro etapas com os valores obtidos. Todas as quatro são executadas,
ainda que a primeira produza achado: é a decisão de não bloquear, cuja consequência se
manifesta no percurso.

---

#### Etapa 1 — ortografia

$$E_1(s) = \bigl\{\, t \;:\; \ell(t) \notin D,\ \ \mathrm{pos}(t) \neq \texttt{PROPN},\ \ \ell(t) \notin \mathbb{N} \,\bigr\}$$

A avaliação é feita **sobre um token por vez**, contra o conjunto:

| $\ell(t)$ | $\ell(t) \in D$ ? | veredito |
|---|:---:|---|
| `há` | sim | — |
| `tres` | **sim** | — |
| `dias` | sim | — |
| `o` | sim | — |
| `incêndio` | sim | — |
| `atingiu` | sim | — |
| `uma` | sim | — |
| `áreas` | sim | — |
| `de` | sim | — |
| `mata` | sim | — |
| `natvia` | **não** | $\in E_1(s)$ |

$$E_1(s) = \{\,\texttt{natvia}\,\}$$

As duas guardas da fórmula — `PROPN` e $\mathbb{N}$ — não são exercitadas neste caso,
dado que a sentença não contém nome próprio nem numeral. Sua função é impedir que
*Chapada dos Veadeiros* e *1.284* sejam detectados como erro. (O código comporta uma
terceira exclusão, o conjunto `TERMOS_DO_DOMINIO` — vazio por padrão —, gancho de
especialização que a fórmula omite por não integrar a definição da técnica.)

A segunda linha merece exame. A forma `tres` está grafada sem acento e ainda assim
**pertence a $D$**, com $f = 1{.}690$ ocorrências. A lista de frequência foi construída a
partir de texto autêntico da *web*, em que a forma sem acento ocorre com frequência
suficiente para ser incorporada. O predicado $\ell(t) \in D$ é verdadeiro, e a etapa não
produz achado.

Não se trata de defeito de implementação, mas do comportamento previsto pela fórmula: $D$
**constitui** a definição operacional de erro, e nada além dela.

Para o achado detectado, a segunda etapa da técnica computa os candidatos e resolve o
desempate por frequência:

$$C(\texttt{natvia}) = \{\texttt{nativa},\ \texttt{naevia},\ \texttt{natia},\ \texttt{latvia}\}$$
$$\hat{w} = \arg\max_{c \in C} f(c) = \texttt{nativa} \quad (f = 912)$$

---

#### Etapa 2 — estrutura

$$\mathrm{pred}(s) \iff
\underbrace{\bigl[\mathrm{pos}(\rho) \in \{\texttt{VERB},\texttt{AUX}\} \wedge \mathrm{VerbForm}(\rho) = \texttt{Fin}\bigr]}_{\text{raiz finita}}
\ \vee\
\underbrace{\exists\, c \in \mathrm{apoio}(\rho) : \mathrm{VerbForm}(c) = \texttt{Fin}}_{\text{auxiliar ou cópula finito}}$$

Não há tabela a apresentar, dado que não há percurso a executar: **inspeciona-se um único
nó**.

```
rho(s)   = atingiu
pos      = VERB           in {VERB, AUX}    -> verdadeiro
VerbForm = Fin                              -> verdadeiro
apoio    = nenhum
```

Satisfeita a primeira cláusula, a segunda não é avaliada.

$$\mathrm{pred}(s) = \text{verdadeiro} \quad\Longrightarrow\quad E_2(s) = \varnothing$$

---

#### Etapa 3 — gramática

$$E_3(s) = \bigl\{\, (t,h,\tau) \;:\; (t,r,h) \in \hat A,\ \tau \in T(r),\
\hat\mu_\tau(t) \neq \bot,\ \hat\mu_\tau(h) \neq \bot,\ \hat\mu_\tau(t) \neq \hat\mu_\tau(h) \,\bigr\}$$

Percorrem-se os arcos, e cada um se desdobra nos traços exigidos por $T(r)$:

| arco $(t,r,h)$ | $T(r)$ | $\tau$ | $\hat\mu_\tau(t)$ | $\hat\mu_\tau(h)$ | veredito |
|---|---|---|---|---|---|
| (`o`, det, `incêndio`) | {Number, Gender} | Number | Sing | Sing | iguais |
| | | Gender | Masc | Masc | iguais |
| (`incêndio`, nsubj, `atingiu`) | {Number, Person} | Number | Sing | Sing | iguais |
| | | Person | $\bot$ | 3 | **comparação suprimida** |
| (`uma`, det, `áreas`) | {Number, Gender} | Number | **Sing** | **Plur** | $\in E_3$ |
| | | Gender | Fem | Fem | iguais |

$$E_3(s) = \bigl\{\, (\texttt{uma},\ \texttt{áreas},\ \texttt{Number}) \,\bigr\}$$

A guarda $\hat\mu_\tau \neq \bot$ opera na quarta linha: o substantivo `incêndio` não
porta traço de `Person`, de modo que a comparação com o `Person=3` do verbo é
**suprimida** — o resultado não é de igualdade, mas de ausência de termos a comparar. Sem
essa guarda, todo sujeito nominal seria computado como discordante quanto à pessoa.

**Apenas três arcos da sentença são admitidos**, e a tabela lhes destina seis linhas. Os
sete arcos restantes não figuram porque $T(r) = \varnothing$: `de` vincula-se por `case`,
`mata` por `nmod` e `áreas` por `obj`, e nenhuma dessas relações impõe concordância.

---

#### O efeito da forma desconhecida sobre a árvore

Convém examinar, antes de prosseguir, os arcos que a etapa 3 **não** inspecionou, dois
dos quais se encontram deformados:

```
(tres,   obj,       Há)         'tres' virou objeto de 'Há'
(dias,   flat:name, tres)       'tres dias' foi lido como NOME PRÓPRIO composto
(natvia, dep,       atingiu)    'dep' é o rótulo de "não sei que relação é esta"
```

A forma `tres` sem acento levou o analisador a interpretar *tres dias* como nome
composto, donde o rótulo `flat:name`. A forma `natvia`, ausente do vocabulário, recebeu
`dep`, rótulo de recurso empregado pelo analisador quando a relação não é classificável.

**Um desvio de grafia não permanece contido na etapa 1.** Ele deforma $\hat A$, e as
etapas 2 e 3 operam sobre a árvore deformada. É essa a razão de o laudo registrar
`[árvore sob suspeita: natvia]` nessas duas etapas — marcação que não indica incorreção,
mas menor confiabilidade.

No caso presente a deformação não atingiu o arco relevante: `(uma, det, áreas)`
permaneceu íntegro e o achado é legítimo. Trata-se, contudo, de contingência posicional,
e não de garantia.

---

#### Etapa 4 — semântica

$$m = \ln P\bigl(\hat{v} \mid s_{\setminus\rho}\bigr) - \ln P\bigl(v_\rho \mid s_{\setminus\rho}\bigr)$$

Igualmente sem tabela: inspeciona-se **uma única** posição, a da raiz. Procede-se ao
mascaramento:

```
Há tres dias o incêndio [MASK] uma áreas de mata natvia.
```

e à leitura da distribuição naquela posição:

```
esperados: ['destruiu', 'atingiu', 'invadiu', 'atinge', 'em']
escrito:   'atingiu'   -> 2ª colocação
margem = 0,95
```

$$E_4(s) = \varnothing \quad\text{porque}\quad m = 0{,}95 < \theta = 5{,}0$$

Manifesta-se aqui o único limiar de toda a pipeline. As três etapas anteriores decidem
por pertinência a um conjunto ou por igualdade de valores, sem parâmetro a calibrar. Esta
compara um valor real a um ponto de corte, e $\theta$ constitui escolha do analista. É a
fronteira entre o que se verifica e o que se estima.

---

#### O laudo

Reunindo os quatro conjuntos pela projeção:

$$\mathrm{Loc}(s) =
\underbrace{\bigl\{(\{\texttt{natvia}\},\,1)\bigr\}}_{E_1}
\ \cup\ \underbrace{\varnothing}_{E_2}
\ \cup\ \underbrace{\bigl\{(\{\texttt{uma},\,\texttt{áreas}\},\,3)\bigr\}}_{E_3}
\ \cup\ \underbrace{\varnothing}_{E_4}$$

$$\mathrm{Loc}(s) \neq \varnothing \quad\Longrightarrow\quad \text{sentença reprovada}$$

Dois achados, cada qual identificando **os tokens envolvidos** e **a etapa de origem**. O
veredito decorre da **vacuidade de um conjunto**: nenhum valor foi somado, ponderado ou
confrontado com limiar agregado. O único limiar existente, o $\theta$ da etapa 4,
determinou apenas a admissão daquele achado, sem participar do veredito final.


<a id="t10-surpresa"></a>

## Surpresa por token

O SLOR e o PLL devolvem **um valor para a sentença integral**. Prestam-se à ordenação de
sentenças, mas não localizam o defeito. A surpresa devolve **um valor por token**: é a
mesma grandeza, examinada antes da agregação.

### A definição

Para cada posição, a surpresa é o logaritmo negativo da probabilidade atribuída pelo
modelo à forma efetivamente ocorrente:

$$\mathrm{surp}(w_i) = -\ln P\bigl(w_i \mid w_{i-1}\bigr)$$

Como $P \in (0,1]$, o logaritmo é negativo e a surpresa é sempre $\geq 0$. Assume valor
$0$ sob certeza do modelo e cresce ilimitadamente à medida que a forma se torna
improvável.

A unidade é relevante. Em logaritmo natural, a grandeza é expressa em **nats**; sob
$\log_2$, em **bits**, caso em que a leitura é literal: a quantidade de informação
veiculada pela forma, ou o número de bits necessários à sua transmissão sob esse modelo.
A surpresa corresponde exatamente à **autoinformação** da teoria da informação.

### A relação com o SLOR

Não constitui técnica nova, mas a parcela que o SLOR já agregava. Retomando a coluna
`contrib` da seção 4:

$$\mathrm{contrib}_i = \ln P_{\mathrm{bi}}(w_i) - \ln P_{\mathrm{uni}}(w_i)
= \mathrm{surp}_{\mathrm{uni}}(w_i) - \mathrm{surp}_{\mathrm{bi}}(w_i)$$

e, consequentemente,

$$\mathrm{SLOR}(s) = \frac{1}{|s|}\sum_{i} \Bigl[\,\mathrm{surp}_{\mathrm{uni}}(w_i)
- \mathrm{surp}_{\mathrm{bi}}(w_i)\,\Bigr]$$

O SLOR é a **média da redução de surpresa produzida pelo contexto**. A técnica desta
seção é o mesmo cálculo, exposto antes da agregação. Nada é recomputado: emprega-se o
estimador já construído, sem modelo adicional nem *download*.

### O comportamento esperado

A expectativa é que a surpresa **se eleve na posição defeituosa**, permitindo a
localização do token. A leitura consiste na identificação do **pico**: a posição de maior
surpresa constitui a candidata a defeito.

Convém registrar desde já a suposição implícita nessa leitura, que o exemplo submeterá a
teste: assume-se a equivalência entre *improvável* e *incorreto*.


In [12]:
import math, re


def surpresa(frase):
    ws = re.findall(r"\w+", frase.lower())
    prev, linhas = INICIO, []
    for w in ws:
        linhas.append((w, -math.log(p_bi(prev, w)), -math.log(p_uni(w)), bi[(prev, w)]))
        prev = w

    pico = max(linhas, key=lambda x: x[1])
    print(frase)
    print(f"   {'token':<15}{'surpresa':>9}{'só unig.':>10}{'c(bi)':>7}")
    for w, s, su, c in linhas:
        print(f"   {w:<15}{s:>9.2f}{su:>10.2f}{c:>7}"
              f"{'   <<< pico' if w == pico[0] and s == pico[1] else ''}")
    print(f"   pico = {pico[1]:.2f} nats em '{pico[0]}'\n")


print("--- onde o contexto ajuda ---")
surpresa("Há três dias o incêndio atingiu uma área de mata nativa")

print("--- frase correta, vocabulário do domínio ---")
surpresa("Focos recorrentes apresentaram persistência anômala")

print("--- a anomalia semântica ---")
surpresa("Há três dias o incêndio cantou uma área de mata nativa")

--- onde o contexto ajuda ---
Há três dias o incêndio atingiu uma área de mata nativa
   token           surpresa  só unig.  c(bi)
   há                  5.54      6.72    259
   três                3.92      7.10     36
   dias                3.73      7.39     30
   o                   3.23      2.86     21
   incêndio            8.25     10.41     22
   atingiu            11.47     10.27      0   <<< pico
   uma                 6.19      4.98      0
   área                6.37      8.16     17
   de                  1.09      2.49    135
   mata               10.00      9.99      4
   nativa              3.53     11.58      2
   pico = 11.47 nats em 'atingiu'

--- frase correta, vocabulário do domínio ---
Focos recorrentes apresentaram persistência anômala
   token           surpresa  só unig.  c(bi)
   focos              13.47     12.27      0
   recorrentes        13.98     12.78      0
   apresentaram       11.65     10.45      0
   persistência       13.47     12.27      0
   an

### Lendo a saída

A coluna `surpresa` corresponde ao resultado da técnica; a coluna `só unig.` registra o
custo da forma **na ausência de contexto**. A diferença entre ambas quantifica a
contribuição do contexto.

**Nas posições em que o contexto intervém, a leitura é adequada.** Na sentença de
referência:

```
nativa      3,53   contra   11,58 sem contexto   →  redução de 8,04 nats
dias        3,73   contra    7,39               →  3,67
três        3,92   contra    7,10               →  3,18
de          1,09   contra    2,49               →  1,40
```

Isoladamente, `nativa` é forma de frequência muito baixa; na sequência de `mata`, contudo,
é praticamente esperada, e a surpresa decresce a menos de um terço. É o comportamento
previsto pela técnica: mensurar a forma **em posição**, e não em abstrato.

### A identidade que explica o restante da tabela

Sempre que $c(bi) = 0$, a surpresa equivale à unigrama acrescida de uma constante:

```
atingiu     11,47  =  10,27 + 1,20
cantou      13,70  =  12,49 + 1,20
anômala     15,08  =  13,88 + 1,20
```

Não se trata de coincidência. Quando o bigrama não foi observado, a interpolação recua
para a unigrama e $P(w_i \mid w_{i-1}) = (1-\lambda)P(w_i)$, donde

$$\mathrm{surp}(w_i) = -\ln\bigl[(1-\lambda)P(w_i)\bigr]
= \mathrm{surp}_{\mathrm{uni}}(w_i) - \ln(1-\lambda)
= \mathrm{surp}_{\mathrm{uni}}(w_i) + 1{,}204$$

**Nessas posições, a surpresa reduz-se à frequência da forma.** O contexto não intervém
no cálculo.

### A regra do pico, reduzida

A leitura proposta pela técnica é um $\arg\max$:

$$\hat{\imath} \;=\; \arg\max_{i} \; \mathrm{surp}(w_i)$$

Cabe determinar o que esse $\arg\max$ efetivamente seleciona. Seja $Z$ o conjunto das
posições degeneradas, aquelas em que o bigrama não foi observado:

$$Z = \{\, i \;:\; c(w_{i-1}, w_i) = 0 \,\}$$

Sobre $Z$, a identidade acima é válida, e sua substituição no interior do $\arg\max$
elimina a constante:

$$\arg\max_{i \in Z} \mathrm{surp}(w_i)
= \arg\max_{i \in Z} \bigl[\mathrm{surp}_{\mathrm{uni}}(w_i) + 1{,}204\bigr]
= \arg\max_{i \in Z} \mathrm{surp}_{\mathrm{uni}}(w_i)$$

E $\mathrm{surp}_{\mathrm{uni}}$ é função **estritamente decrescente** da contagem:

$$\mathrm{surp}_{\mathrm{uni}}(w) = -\ln\frac{c(w)+1}{N+V}
\qquad\Longrightarrow\qquad
\arg\max_{i \in Z} \mathrm{surp}_{\mathrm{uni}}(w_i) = \arg\min_{i \in Z} c(w_i)$$

$$\boxed{\;\text{sobre } Z, \quad \hat{\imath} \;=\; \arg\min_i c(w_i)\;}$$

**O pico corresponde à forma de menor frequência da sentença.** Não se trata de
tendência, mas de identidade, válida sempre que a posição pertença a $Z$. A forma
antecedente não intervém no cálculo, tampouco a ordem ou a estrutura: resta a frequência
isolada, e o $\arg\max$ seleciona seu mínimo.

#### A verificação, com o desempate

Em *Focos recorrentes apresentaram persistência anômala*, **todas** as cinco posições
pertencem a $Z$, de modo que a redução se aplica à sentença integral. Confrontam-se as
duas ordenações:

| $w$ | $c(w)$ | posto por $c$ ↑ | surpresa | posto por surpresa ↓ |
|---|---:|:---:|---:|:---:|
| `anômala` | 0 | 1º | 15,08 | 1º |
| `recorrentes` | 2 | 2º | 13,98 | 2º |
| `focos` | 4 | 3º (empate) | **13,47** | 3º (empate) |
| `persistência` | 4 | 3º (empate) | **13,47** | 3º (empate) |
| `apresentaram` | 30 | 5º | 11,65 | 5º |

As duas colunas de posto coincidem. O **empate corrobora a identidade de modo mais
convincente que a ordenação**: `focos` e `persistência` são formas distintas, em posições
distintas e com antecessores distintos, e recebem surpresa **idêntica até a segunda casa
decimal**, por partilharem $c(w) = 4$. Caso o contexto interviesse em alguma medida, os
dois valores divergiriam. Não divergem.

Recomputando o valor para uma delas, sem consulta ao modelo:

$$-\ln\frac{4+1}{1{.}065{.}776} + 1{,}204 \;=\; 12{,}270 + 1{,}204 \;=\; 13{,}47$$

#### O pico da sentença correta

Na sentença de referência a redução se aplica apenas parcialmente, dado que $Z$ possui
somente dois elementos:

| $i \in Z$ | $w_i$ | $c(w_i)$ | surpresa |
|---|---|---:|---:|
| 6 | `atingiu` | 36 | **11,47** |
| 7 | `uma` | 7.305 | 6,19 |

Entre os dois, $\arg\min c$ corresponde a `atingiu`. Os nove tokens **externos** a $Z$ —
aqueles em que o contexto efetivamente interveio — não excedem $10{,}00$ (`mata`). O
máximo global é, portanto, `atingiu`, com $11{,}47$.

Reunidos os dois resultados, o comportamento da regra do pico torna-se explícito: em uma
sentença **correta**, ela localiza o token cujo bigrama era ausente no corpus **e** que,
entre os ausentes, apresentava menor frequência. Ambas as condições são propriedades do
corpus de treino; a sentença não participa da seleção.

### O que isso produz — dois alarmes falsos e um acerto

| frase | pico | posição | desvio? |
|---|---:|---|---|
| Há três dias o incêndio **atingiu** uma área de mata nativa. | $11{,}47$ | `atingiu` | **não** |
| Focos recorrentes apresentaram persistência **anômala**. | $15{,}08$ | `anômala` | **não** |
| Há três dias o incêndio **cantou** uma área de mata nativa. | $13{,}70$ | `cantou` | sim |

Na **sentença correta**, o pico recai sobre `atingiu` — o verbo adequado, na posição
adequada. É localizado apenas porque o bigrama `incêndio atingiu` não ocorre no corpus.

Mais grave: *Focos recorrentes apresentaram persistência anômala* é português técnico
correto, emprega três termos que nomeiam colunas da própria base e **atinge pico de
$15{,}08$, superior ao da anomalia efetiva**.

A consequência prática é relevante: **qualquer limiar que detecte `cantou` detecta
previamente `anômala` e, muito provavelmente, também `atingiu`**. Não há ponto de corte
que capture o desvio sem capturar antes as sentenças corretas.

### O que a técnica não distingue

A leitura que identifica pico com defeito pressupõe a coincidência entre improvável e
incorreto. Tal coincidência não se verifica. Recebem surpresa elevada, de forma
indistinguível:

- o **incorreto** — a forma inadequada à posição;
- o **raro** — vocabulário técnico, nome próprio, termo estrangeiro;
- o **original** — a formulação inesperada, que em texto bem redigido constitui
  qualidade;
- o **não observado pelo estimador** — tudo o que o corpus não registra, que é a maior
  parte.

A técnica mensura o último. Os três primeiros não são distinguidos entre si. E em um
domínio cujo vocabulário o corpus de 1994 cobre precariamente, a maior parte das
ocorrências recai no último caso.

### Quando usar

- **Para localização, posteriormente à detecção por outra técnica.** É o emprego
  legítimo: não como critério de decisão, mas como indicador. Havendo achado de um
  detector confiável, o perfil de surpresa sugere o ponto de partida da inspeção.
- **Na comparação da mesma posição entre variantes.** Fixado o contexto e alterada
  apenas a forma, a comparação é legítima, dado que a frequência de base é a única
  variável e essa condição é conhecida.
- **Como diagnóstico do modelo, e não do texto.** O perfil evidencia o que o estimador
  registrou e o que não observou, constituindo meio adequado para constatar
  que o corpus é insuficiente para a tarefa.

### Quando não usar

- **Como detector, com limiar absoluto.** É o equívoco central, e a tabela dos três picos
  explicita a razão: não há valor de corte que separe desvio de raridade, uma vez que a
  escala não é absoluta, mas dependente do vocabulário da sentença.
- **Na comparação de picos entre sentenças de conteúdos distintos.** Uma sentença técnica
  apresentará picos elevados por construção. Confrontar seu pico com o de uma sentença
  corrente equivale a comparar vocabulários, e não qualidade.
- **Sobre estimador esparso, sob expectativa de informação contextual.** Com bigramas
  computados em corpus de um milhão de tokens, a maior parte das posições recai na
  identidade acima e devolve frequência sob a aparência de surpresa. O sintoma é a coluna
  `c(bi)` integralmente nula.
- **Sobre o primeiro token, na ausência de marcador de início no modelo.** Sem contexto à
  esquerda, essa posição não é comparável às demais. No caso presente ela o é, porque
  `<s>` dispõe de contagem própria — decisão de implementação, e não propriedade do dado.


<a id="t11-verbo"></a>

## Plausibilidade do verbo por preenchimento mascarado (BERTimbau)

O SLOR e o PLL avaliam a probabilidade da sentença; a surpresa por token avalia a
probabilidade de cada forma em sua posição, mas o faz examinando apenas a forma escrita.
Esta técnica inverte o procedimento: suprime-se a forma e determina-se **qual forma seria
esperada naquela posição**.

A diferença é substancial. Em lugar de atribuir escore ao que está escrito, obtém-se a
**expectativa do modelo para aquela posição**, e somente então se confronta o texto com
ela.

### A posição a suprimir

A supressão de posição arbitrária não seria adequada. A escolha é sintática: suprime-se a
**raiz** da árvore de dependências, isto é, o verbo que sustenta a predicação e o nó que
seleciona os demais argumentos. A técnica só se aplica quando

$$\mathrm{pos}\bigl(\rho(s)\bigr) = \texttt{VERB}$$

e, quando essa condição é falsa — fragmento, predicação nominal —, a resposta adequada é
`n/a`, e não um escore. Não se trata de limitação implícita, mas de elemento
constitutivo da definição.

Cabe observar o arranjo: o **analisador sintático** determina *a posição* a ser
consultada; o **modelo estatístico** determina *o que é plausível* naquela posição.
Nenhum dos dois executaria a tarefa isoladamente.

### A medida

Substituída a raiz pela marca, obtém-se $s_{\setminus\rho}$, e o modelo devolve uma
distribuição sobre todo o vocabulário $\mathcal{V}$:

$$P\bigl(v \mid s_{\setminus\rho}\bigr), \qquad
\sum_{v \in \mathcal{V}} P\bigl(v \mid s_{\setminus\rho}\bigr) = 1$$

Cabe sublinhar: **trata-se de probabilidade em sentido estrito**, que soma 1 sobre uma
posição. Diferentemente do PLL, não há aqui nada de *pseudo*: a soma percorre as
alternativas de uma única posição, que é exatamente o que o modelo mascarado foi treinado
para prever.

Extraem-se dela duas leituras. A **absoluta** é o logaritmo da probabilidade do verbo
escrito:

$$\ln P\bigl(v_\rho \mid s_{\setminus\rho}\bigr)$$

E a **margem**, distância até o melhor preenchimento possível:

$$\hat{v} = \arg\max_{v \in \mathcal{V}} P\bigl(v \mid s_{\setminus\rho}\bigr)
\qquad\qquad
m(s) = \ln P\bigl(\hat{v} \mid s_{\setminus\rho}\bigr) - \ln P\bigl(v_\rho \mid s_{\setminus\rho}\bigr)$$

Por construção $m(s) \geq 0$, e $m(s) = 0$ exatamente quando o verbo escrito **é** a
primeira escolha do modelo.

### Por que a margem, e não o valor absoluto

O valor absoluto depende do grau geral de previsibilidade da posição. Em contexto muito
restrito, toda alternativa recebe probabilidade elevada; em contexto vago, toda
alternativa recebe probabilidade baixa, inclusive o verbo adequado. Comparar $\ln P$
entre contextos distintos agrega as duas grandezas.

A margem normaliza pelo **máximo daquela posição**. Quantifica a distância ao melhor
preenchimento, e não a probabilidade absoluta, sendo por isso comparável entre sentenças.


In [13]:
import torch, spacy
from transformers import AutoTokenizer, AutoModelForMaskedLM

nlp = spacy.load("pt_core_news_md")
MODELO = "neuralmind/bert-base-portuguese-cased"
tok = AutoTokenizer.from_pretrained(MODELO)
mlm = AutoModelForMaskedLM.from_pretrained(MODELO)
mlm.eval()

@torch.no_grad()
def verbo_mascarado(frase, k=5):
    doc = nlp(frase)
    raiz = [t for t in doc if t.dep_ == "ROOT"][0]
    print(frase)

    if raiz.pos_ != "VERB":                       # a técnica não se aplica
        print(f"   n/a — a raiz é '{raiz.text}' ({raiz.pos_}), não um verbo\n")
        return

    ids_alvo = tok(raiz.text, add_special_tokens=False)["input_ids"]
    if len(ids_alvo) != 1:                        # verbo partido em subpalavras
        print(f"   n/a — '{raiz.text}' não é uma peça única do vocabulário\n")
        return

    mascarada = "".join((tok.mask_token if t.i == raiz.i else t.text) + t.whitespace_
                        for t in doc)
    enc = tok(mascarada, return_tensors="pt")
    pos = int((enc["input_ids"][0] == tok.mask_token_id).nonzero()[0])
    lp = torch.log_softmax(mlm(**enc).logits[0, pos], dim=-1)

    escrito = float(lp[ids_alvo[0]])
    topo = torch.topk(lp, k)
    melhor = float(topo.values[0])

    print(f"   raiz mascarada: '{raiz.text}'")
    print(f"   o modelo esperava: {[tok.convert_ids_to_tokens([int(i)])[0] for i in topo.indices]}")
    print(f"   ln P(escrito) = {escrito:7.2f}    ln P(melhor) = {melhor:7.2f}"
          f"    margem = {melhor - escrito:5.2f}\n")

print("--- par mínimo: mesmo contexto, verbos diferentes ---")
verbo_mascarado("Há três dias o incêndio atingiu uma área de mata nativa.")
verbo_mascarado("Há três dias o incêndio cantou uma área de mata nativa.")

print("--- quando a técnica não se aplica ---")
verbo_mascarado("Há três dias o incêndio que atingiu uma área de mata nativa.")
verbo_mascarado("O incêndio é intenso.")
verbo_mascarado("Há três dias o incêndio destruiu uma área de mata nativa.")


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 35791.49it/s]
[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- par mínimo: mesmo contexto, verbos diferentes ---
Há três dias o incêndio atingiu uma área de mata nativa.
   raiz mascarada: 'atingiu'
   o modelo esperava: ['destruiu', 'atingiu', 'atinge', 'invadiu', 'em']
   ln P(escrito) =   -0.91    ln P(melhor) =   -0.72    margem =  0.20

Há três dias o incêndio cantou uma área de mata nativa.
   raiz mascarada: 'cantou'
   o modelo esperava: ['destruiu', 'atingiu', 'atinge', 'invadiu', 'em']
   ln P(escrito) =  -14.45    ln P(melhor) =   -0.72    margem = 13.73

--- quando a técnica não se aplica ---
Há três dias o incêndio que atingiu uma área de mata nativa.
   n/a — a raiz é 'dias' (NOUN), não um verbo

O incêndio é intenso.
   n/a — a raiz é 'intenso' (ADJ), não um verbo

Há três dias o incêndio destruiu uma área de mata nativa.
   raiz mascarada: 'destruiu'
   o modelo esperava: ['destruiu', 'atingiu', 'atinge', 'invadiu', 'em']
   ln P(escrito) =   -0.72    ln P(melhor) =   -0.72    margem =  0.00



### Lendo a saída

| frase | $\ln P$(escrito) | margem |
|---|---:|---:|
| Há três dias o incêndio **atingiu** uma área de mata nativa. | $-0{,}91$ | $0{,}20$ |
| Há três dias o incêndio **cantou** uma área de mata nativa. | $-14{,}45$ | $\mathbf{13{,}73}$ |
| Há três dias o incêndio **destruiu** uma área de mata nativa. | $-0{,}72$ | $0{,}00$ |

A separação é ampla: quase 14 nats entre o verbo adequado e o anômalo, enquanto os dois
verbos plausíveis se situam junto ao zero.

O aspecto metodologicamente relevante é a lista de formas esperadas, **idêntica** nas
três sentenças: `['destruiu', 'atingiu', 'atinge', 'invadiu', 'em']`.

E não poderia ser distinta. Ao mascarar a raiz, suprime-se precisamente o único token em
que as sentenças diferem; o contexto remanescente é o mesmo, e portanto a expectativa do
modelo é a mesma. **O instrumento de medida permanece fixo, variando apenas o objeto
mensurado.** É essa a condição que torna a comparação legítima: a diferença entre
$0{,}20$ e $13{,}73$ não é atribuível a nenhum outro fator.

A terceira linha exibe o piso da medida. A forma `destruiu` é a **primeira escolha** do
modelo, e por isso a margem é exatamente $0{,}00$. O valor serve de calibração: a margem
não constitui escala arbitrária, mas possui zero dotado de significado — a coincidência
entre a forma escrita e a forma esperada.

A forma `atingiu`, com $0{,}20$, é a segunda escolha. Duas formulações igualmente
adequadas do mesmo fato recebem margens quase idênticas, propriedade desejável em uma
medida de plausibilidade.

### A distribuição, exibida

O fundamento acima estabelece que o modelo devolve distribuição que soma 1 sobre
$\mathcal{V}$. A saída exibe apenas cinco rótulos; convém examinar a massa de
probabilidade que concentram.

$$|\mathcal{V}| = 29{.}794
\qquad
\sum_{v \in \mathcal{V}} P\bigl(v \mid s_{\setminus\rho}\bigr) = 1$$

| posto | $v$ | $\ln P$ | $P$ | acumulado |
|---:|---|---:|---:|---:|
| 1 | `destruiu` | $-0{,}72$ | 0,4889 | 0,4889 |
| 2 | `atingiu` | $-0{,}91$ | 0,4007 | **0,8896** |
| 3 | `atinge` | $-3{,}06$ | 0,0467 | 0,9363 |
| 4 | `invadiu` | $-3{,}17$ | 0,0418 | 0,9781 |
| 5 | `em` | $-5{,}16$ | 0,0058 | **0,9839** |
| ⋮ | | | | |
| 29.794 | | | | 1,0000 |

**Dois tokens concentram 89% da massa; cinco concentram 98,4%.** Os 29.789 restantes
repartem 1,6%. Esse resultado legitima a leitura da distribuição como expectativa: neste
contexto a expectativa é efetiva e se concentra em duas formas. Tal concentração não se
verifica necessariamente em todos os contextos, e é a própria tabela que permite
constatá-la, posição a posição.

### A margem é uma razão, e o máximo intervém nela

$$m(s) = \ln P(\hat{v}) - \ln P(v_\rho) = \ln \frac{P(\hat{v})}{P(v_\rho)}$$

Para `cantou`:

$$m = \ln \frac{0{,}4889}{5{,}33 \times 10^{-7}} = \ln\bigl(917{.}000\bigr) = 13{,}73$$

Cabe considerar, a seguir, o dado que recalibra a leitura:

| $v$ | $P$ | posto em $\mathcal{V}$ | margem |
|---|---:|---:|---:|
| `destruiu` | $4{,}89\times10^{-1}$ | 1 | 0,00 |
| `atingiu` | $4{,}01\times10^{-1}$ | 2 | 0,20 |
| `construiu` | $1{,}68\times10^{-6}$ | 256 | 12,58 |
| `cantou` | $5{,}33\times10^{-7}$ | **420** | 13,73 |

**A forma `cantou` ocupa a posição 420 entre 29.794.** Há 29.374 tokens menos prováveis
que ela nesta posição, o que a situa no percentil 98,6 do vocabulário. A margem de 13,73
sugere distância considerável; o posto sugere apenas plausibilidade mediana.

As duas leituras não são contraditórias, e a razão reside na fórmula. A grandeza $m$ é o
logaritmo de um quociente cujo numerador é $P(\hat{v}) = 0{,}489$. Quando o máximo
concentra metade da massa, **todas as demais alternativas ficam logaritmicamente
distantes por consequência aritmética**. Caso o contexto fosse vago e o máximo valesse
$0{,}05$, a mesma forma `cantou` produziria

$$\ln \frac{0{,}05}{5{,}33 \times 10^{-7}} = 11{,}45$$

isto é, dois nats e meio a menos, sem alteração alguma na forma escrita. O fundamento
acima estabelece que a margem normaliza pelo máximo daquela posição, o que é correto; cabe
acrescentar que essa normalização faz **o próprio valor do máximo intervir no resultado**.
Margens provenientes de contextos com concentrações distintas não constituem grandezas de
mesma escala.


### O que a medida captura

A sentença anômala não apresenta desvio de grafia, de concordância ou de estrutura. *Há
três dias o incêndio cantou uma área de mata nativa* possui árvore bem formada, e todas
as formas constam do léxico. O desvio consiste em violação de **restrição de seleção**:
*cantar* não admite *área* como paciente, e *incêndio* não é agente admissível de
*cantar*.

É a primeira técnica examinada que alcança esse nível. O léxico não o detecta, tampouco o
analisador de traços ou as regras; as medidas globais o confundem com raridade — a
surpresa por token chegou a atribuir pico superior a uma sentença correta.

### Os casos de não aplicabilidade

```
Há três dias o incêndio que atingiu uma área de mata nativa.
   n/a — a raiz é 'dias' (NOUN), não um verbo

O incêndio é intenso.
   n/a — a raiz é 'intenso' (ADJ), não um verbo
```

Ambas as respostas são `n/a`, e ambas estão corretas. A primeira corresponde ao
fragmento: sem verbo na raiz não há o que mascarar, constatação que o teste da raiz, na
seção 6, já fornecia a custo inferior. A segunda corresponde à predicação nominal: o verbo
`é` está presente, mas como cópula, e não como raiz.

O reconhecimento da não aplicabilidade **integra o emprego correto da técnica**. Uma
implementação que devolvesse um valor nesses casos estaria mascarando um token arbitrário
e apresentando o resultado como resposta à questão originalmente formulada.


#### A terceira causa de `n/a`, e sua magnitude

Há um terceiro motivo, tratado pelo código e não exibido pelo exemplo: o verbo pode não
constituir peça única do vocabulário. A magnitude do fenômeno é mensurável. Tomando-se as
formas verbais do mac_morpho:

| conjunto | formas | peça única | |
|---|---:|---:|---:|
| todas as formas distintas | 13.417 | 2.143 | **16,0%** |
| formas com $c \geq 5$ | 3.137 | 1.638 | 52,2% |
| formas com $c \geq 20$ | 906 | 810 | 89,4% |
| 200 formas mais frequentes | 200 | 198 | 99,0% |
| **ponderado por ocorrência** | 118.367 usos | — | **75,0%** |

O resultado varia conforme se contabilizem **tipos** ou **ocorrências**. Por tipo, a
cobertura é de 16%; por uso, de 75%. Ambas as leituras são corretas e respondem a questões
distintas: a fração do léxico verbal alcançada e a fração dos verbos efetivamente
encontrados em texto corrente. Para auditoria de texto corrido, a segunda é a pertinente
— e um quarto dos verbos permanece fora do alcance.

**Parte dessa perda não decorre de raridade, mas de caixa.** O BERTimbau dispõe de
vocabulário *sensível a caixa*, e a mesma forma é segmentada de modos distintos:

```
Atingiu   ['At', '##ing', '##iu']          atingiu   ['atingiu']
Destruiu  ['Des', '##tru', '##iu']         destruiu  ['destruiu']
Tenho     ['Ten', '##ho']                  tenho     ['tenho']
Estamos   ['Esta', '##mos']                estamos   ['estamos']
```

O verbo da sentença de referência, `atingiu`, constitui peça única em minúscula e é
segmentado em **três** peças quando inicia a sentença. Mensurando o efeito sobre as formas
com $c \geq 5$:

| formas | peça única |
|---|---:|
| iniciadas por maiúscula | 65 de 218 (29,8%) |
| minúsculas | 1.573 de 2.919 (53,9%) |

e **97 das 153 formas iniciadas por maiúscula que são segmentadas voltariam a constituir
peça única em minúscula** — 63% delas. Em síntese: parcela considerável dos casos de `n/a`
nada estabelece sobre o verbo, apenas registra que ele ocorreu em posição inicial de
sentença.

A correção não se obtém pela conversão prévia para minúscula: a alteração de caixa
modifica o contexto lido pelo modelo, e há casos de sentido inverso (`Veja` é peça única,
ao passo que `veja` é segmentada em duas). Cabe, contudo, considerar essa componente ao
relatar cobertura, sob pena de se atribuir ao léxico o que é de natureza tipográfica.

### Quando usar

- **Para anomalia de seleção verbal**, com sintaxe íntegra. É o caso para o qual a técnica
  foi concebida, e o único em que constitui o melhor instrumento entre os examinados.
- **Com a margem, e não com limiar absoluto sobre $\ln P$.** A margem é normalizada ao
  máximo da posição, sendo por isso comparável entre sentenças de contextos distintos.
- **Posteriormente às etapas de menor custo.** A técnica pressupõe sentença já bem
  formada: havendo fragmento ou desvio estrutural, a raiz sequer será um verbo e a
  resposta será n/a.
- **Quando se pretende exibir a alternativa.** A lista de formas esperadas é a saída mais
  informativa da técnica: não se limita a assinalar anomalia, mas indica **qual forma
  seria adequada à posição**.

### Quando não usar

- **Fora da raiz.** A técnica inspeciona uma única posição. Em *o incêndio atingiu uma
  área de mata azul* a anomalia recai sobre o objeto, e o verbo é adequado: a margem não
  produziria achado. Cada posição adicional a ser testada acarreta uma passagem a mais
  pelo modelo.
- **Quando a raiz não é verbo.** Fragmentos e predicações nominais devolvem n/a. A técnica
  não constitui detector geral, mas teste sobre um nó específico, e o reconhecimento de
  sua não aplicabilidade integra o emprego correto.
- **Quando o verbo não constitui peça única do vocabulário.** É restrição concreta e
  frequente: `brincaram`, `analisaram` e `beberam` são todos segmentados em subpalavras
  pelo BERTimbau, e nesses casos $\ln P$ do verbo não existe como valor único. Comparar
  apenas a primeira subpalavra é aproximação que altera a grandeza mensurada; é preferível
  devolver n/a a devolver um valor com aparência de probabilidade que não o é.
- **Como critério de correção.** A medida é de **tipicidade de uso**, aprendida de corpus.
  Uma construção legítima porém incomum recebe margem elevada, e uma sentença falsa porém
  trivial recebe margem baixa. O caso mais evidente consta da própria tabela desta seção:
  `destruiu` é a **primeira escolha** do modelo, com margem $0{,}00$, e `atingiu` figura
  em segundo, com $0{,}20$. Caso o incêndio tenha alcançado a área sem destruí-la,
  `atingiu` é o verbo verdadeiro e `destruiu` o falso — e a medida atribui escore
  **superior** ao falso. Plausibilidade não equivale a verdade, e neste caso as duas se
  opõem.
- **Sem verificação da raiz.** A posição é determinada pelo analisador. Estando a árvore
  incorreta, a máscara recai sobre token diverso, e o valor resultante não responde à
  questão pretendida.


<a id="t12-embeddings"></a>

## Embeddings

Todas as técnicas anteriores tratavam a forma como **símbolo**: pertence ou não ao
léxico, casa ou não com a regra, possui ou não contagem no corpus. Duas formas distintas
eram simplesmente distintas, sem gradação.

Esta técnica substitui o símbolo por um **ponto no espaço**, obtendo com isso uma noção
de proximidade ausente nas anteriores.

### A origem do vetor

O fundamento é a **hipótese distribucional**: formas que ocorrem nos mesmos contextos
possuem significados relacionados. O vetor é construído de modo a sumarizar os contextos
de ocorrência da forma no corpus de treino, e o resultado é uma tabela

$$\mathbf{v} : \mathcal{V} \to \mathbb{R}^{d}$$

com uma linha por forma. No caso presente, $d = 300$ e a tabela possui 20.000 linhas.
Trata-se de **consulta**, e não de cálculo: o vetor de `casa` é invariante, armazenado no
modelo.

Da propriedade seguinte decorrem tanto os acertos quanto as falhas da técnica: **o vetor
não codifica o significado da forma, mas os contextos em que ela habitualmente ocorre.**

### A medida de proximidade

A comparação é feita pelo **cosseno**, isto é, pelo ângulo entre os dois vetores:

$$\cos(\mathbf{a}, \mathbf{b})
= \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a}\rVert \, \lVert \mathbf{b}\rVert}
= \frac{\sum_{i=1}^{d} a_i b_i}
       {\sqrt{\sum_i a_i^2}\;\sqrt{\sum_i b_i^2}}$$

O valor situa-se entre $-1$ e $+1$: assume $1$ quando os vetores apontam na mesma direção
e $0$ quando são ortogonais.

**Justificativa do cosseno em lugar da distância euclidiana.** A norma
$\lVert \mathbf{v} \rVert$ veicula sobretudo frequência: formas comuns tendem a vetores
de maior comprimento. A divisão pelas duas normas descarta essa magnitude e retém apenas
a **direção**, onde reside a informação contextual. Sem essa normalização, `casa` e
`apartamento` apresentariam distância elevada apenas em virtude da diferença de
frequência.

### Da forma para a sentença

A extensão mais simples é a média dos vetores dos tokens:

$$\mathbf{v}(s) = \frac{1}{|s|} \sum_{t \in s} \mathbf{v}(t)$$

Convém examinar essa fórmula antes de adotá-la. O somatório percorre um
**multiconjunto**: qualquer reordenação de $s$ apresenta exatamente os mesmos termos e,
consequentemente, a mesma soma. Portanto, para toda permutação $\pi$,

$$\mathbf{v}\bigl(\pi(s)\bigr) = \mathbf{v}(s)
\qquad\Longrightarrow\qquad
\cos\bigl(\mathbf{v}(s),\, \mathbf{v}(\pi(s))\bigr) = 1$$

Não se trata de aproximação nem de perda tolerável, mas de **identidade**. O exemplo
adiante explicita seu custo.


In [14]:
import spacy
import numpy as np

nlp = spacy.load("pt_core_news_md")
print(f"tabela de vetores: {nlp.vocab.vectors.shape[0]} formas x "
      f"{nlp.vocab.vectors.shape[1]} dimensões\n")

def cosseno(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else float("nan")

def par(x, y):
    print(f"   {x:<10} x {y:<12} cos = {cosseno(nlp.vocab[x].vector, nlp.vocab[y].vector):+.3f}")

print("SINÔNIMOS")
for x, y in [("incêndio", "fogo"), ("mata", "floresta"),
             ("área", "região"), ("chamas", "fogo")]:
    par(x, y)

print("\nANTÔNIMOS")
for x, y in [("seco", "úmido"), ("quente", "frio"),
             ("início", "fim"), ("ativo", "extinto")]:
    par(x, y)

print("\nSEM RELAÇÃO")
for x, y in [("incêndio", "gramática"), ("área", "advérbio"), ("fogo", "sintaxe")]:
    par(x, y)

print("\nFORA DA TABELA")
for p in ["mata", "incêndio", "natvia"]:
    t = nlp.vocab[p]
    print(f"   {p:<16} tem vetor: {str(t.has_vector):<6} norma = {np.linalg.norm(t.vector):.2f}")

def vetor_frase(frase):
    return np.mean([t.vector for t in nlp(frase) if t.has_vector], axis=0)

print("\nFRASE = MÉDIA DOS VETORES")
a, b = "O fogo atingiu o vale.", "O vale atingiu o fogo."
print(f"   {a}\n   {b}\n   cos = {cosseno(vetor_frase(a), vetor_frase(b)):.6f}")


tabela de vetores: 20000 formas x 300 dimensões

SINÔNIMOS
   incêndio   x fogo         cos = +0.509
   mata       x floresta     cos = +0.610
   área       x região       cos = +0.658
   chamas     x fogo         cos = +0.648

ANTÔNIMOS
   seco       x úmido        cos = +0.838
   quente     x frio         cos = +0.690
   início     x fim          cos = +0.562
   ativo      x extinto      cos = +0.263

SEM RELAÇÃO
   incêndio   x gramática    cos = +0.005
   área       x advérbio     cos = -0.063
   fogo       x sintaxe      cos = +0.065

FORA DA TABELA
   mata             tem vetor: True   norma = 38.33
   incêndio         tem vetor: True   norma = 27.67
   natvia           tem vetor: False  norma = 0.00

FRASE = MÉDIA DOS VETORES
   O fogo atingiu o vale.
   O vale atingiu o fogo.
   cos = 1.000000


### Lendo a saída

O bloco *sem relação* estabelece a referência: `incêndio`×`gramática` resulta em
$+0{,}005$ e `área`×`advérbio` em $-0{,}063$. **O zero corresponde à ausência de
relação**, e é contra esse valor que os demais devem ser interpretados.

### O resultado que invalida a leitura ingênua

Ordenando os pares pelo cosseno:

```
seco      x úmido       +0,838     ANTÔNIMOS
quente    x frio        +0,690     ANTÔNIMOS
área      x região      +0,658     sinônimos
chamas    x fogo        +0,648     sinônimos
mata      x floresta    +0,610     sinônimos
início    x fim         +0,562     ANTÔNIMOS
incêndio  x fogo        +0,509     sinônimos
ativo     x extinto     +0,263     ANTÔNIMOS
```

**O par de maior proximidade em toda a tabela é um par de opostos.** O par `seco`/`úmido`
($0{,}838$) supera todos os sinônimos, inclusive `incêndio`/`fogo` ($0{,}509$), cujos
termos designam praticamente a mesma entidade.

Não se trata de ruído nem de defeito do modelo, mas do funcionamento previsto da hipótese
distribucional: `seco` e `úmido` ocorrem exatamente nos mesmos contextos — *clima ___*,
*período ___*, *solo ___*. Uma vez que o vetor sumariza contextos, ambos recebem direções
quase coincidentes. As formas `incêndio` e `fogo`, ao contrário, pertencem a registros
distintos: a primeira designa o evento noticiado, a segunda o elemento.

O cosseno mensura **campo temático**, e não **sentido**. Formas do mesmo campo situam-se
próximas, sejam elas equivalentes ou opostas. Não há limiar capaz de separá-las, uma vez
que a informação necessária a essa distinção nunca foi incorporada ao vetor.

### O cosseno, aberto

$$\cos(\mathbf{a},\mathbf{b}) = \frac{\sum_{i=1}^{300} a_i b_i}{\lVert\mathbf{a}\rVert\,\lVert\mathbf{b}\rVert}$$

Para `incêndio` × `fogo`, os três valores requeridos pela fórmula:

$$\mathbf{a}\cdot\mathbf{b} = 585{,}23
\qquad
\lVert\mathbf{a}\rVert = 27{,}67
\qquad
\lVert\mathbf{b}\rVert = 41{,}58
\qquad
\frac{585{,}23}{27{,}67 \times 41{,}58} = 0{,}509$$

O produto interno é uma soma de **300** parcelas, nenhuma das quais dotada de significado
isolado:

```
a[:5] = [-0,44   1,28  -0,63   1,54   0,89]
b[:5] = [-2,52  -1,16   1,57   1,34   1,11]
        ------------------------------------
         [ 1,11 -1,48  -0,99   2,06   0,99]   soma parcial = 1,69   de 585,23
```

Cinco dimensões contribuem com 1,69 dos 585,23, e com sinais alternados entre si. O valor
0,509 emerge do conjunto das 300; não há dimensão identificável como correspondente ao
calor ou ao desastre. É essa a razão de a técnica não produzir localização: **a unidade
de informação é o vetor integral**, e ele não se decompõe em componentes interpretáveis.

**As normas confirmam o exposto no fundamento.** Formas gramaticais recebem vetores
de maior comprimento; formas de conteúdo, de menor:

| forma | norma | | forma | norma |
|---|---:|---|---|---:|
| `os` | 101,46 | | `área` | 51,43 |
| `Há` | 94,72 | | `cantou` | 32,38 |
| `há` | 85,83 | | `atingiu` | 28,50 |
| `o` | 72,96 | | `incêndio` | 27,67 |

O vetor de `os` é quase quatro vezes mais longo que o de `incêndio`. A divisão pelas duas
normas é o que impede que a comparação se converta em comparação de frequências.

### O corpus fixo através da média

$$\mathbf{v}(s) = \frac{1}{|s|}\sum_{t \in s}\mathbf{v}(t)$$

Aplicando-se a fórmula a cada variante e comparando-se com a sentença de referência:

| variante | o que mudou | $\cos$ com a base |
|---|---|---:|
| ortografia — `natvia` | uma letra trocada | 0,992002 |
| **anomalia — `cantou`** | **o incêndio canta** | **0,987618** |
| conc. nominal — `uma áreas` | um `s` | 0,977767 |
| ordem embaralhada | 11 formas permutadas | 0,966889 |
| conc. verbal — `os incêndios atingiu` | dois `s` | 0,859625 |
| **escolha — `A três dias`** | **`há` virou `a`** | **0,808742** |

As duas linhas em negrito devem ser lidas em conjunto. A sentença em que **um incêndio
canta uma área** é a segunda mais próxima da referência, a $0{,}988$. A sentença em que
`há` foi substituído por `a` — desvio de um único caractere, que não altera fato algum —
é a **menos** próxima de todas, a $0{,}809$. A ordenação encontra-se invertida em relação
à gravidade dos desvios.

### A explicação

Quando $s'$ difere de $s$ por uma substituição $u \to w$, a média se desloca por

$$\mathbf{v}(s') - \mathbf{v}(s) = \frac{\mathbf{v}(w) - \mathbf{v}(u)}{|s|}$$

e a magnitude desse deslocamento depende **exclusivamente da distância entre as duas
linhas da tabela**. Mensurando as substituições do corpus:

| troca | $\lVert\mathbf{v}(w)-\mathbf{v}(u)\rVert$ | dividido por $\lvert s\rvert=12$ | $\cos$ dos dois tokens |
|---|---:|---:|---:|
| `há` → `a` | 110,55 | 9,21 | $-0{,}003$ |
| `o`→`os` **e** `incêndio`→`incêndios` | 127,83 | 10,65 | — |
| `área` → `áreas` | 47,78 | 3,98 | — |
| `atingiu` → `cantou` | 35,57 | 2,96 | $+0{,}323$ |

A última coluna explica o resultado. As formas `a` e `há` são **ortogonais** — cosseno
$-0{,}003$, o mesmo patamar dos pares sem relação. Um artigo e um verbo existencial não
partilham contexto algum, de modo que suas linhas apontam para direções independentes. As
formas `atingiu` e `cantou`, ao contrário, apresentam cosseno $+0{,}323$: ambas são verbos
no pretérito, ocorrem após o sujeito e antes do objeto, e a hipótese distribucional as
aproxima por essa razão.

O vetor médio da sentença de referência apresenta norma $18{,}80$. Um deslocamento de
$9{,}21$ corresponde a quase metade desse valor; um de $2{,}96$, a um sexto. **A anomalia
semântica desloca a média de forma reduzida, ao passo que a substituição de forma
funcional a desloca de forma acentuada** — porque a fórmula pondera a distância na tabela,
e distância na tabela corresponde a similaridade de contexto, e não de sentido.

(O deslocamento explica as magnitudes, mas não a ordenação exata dos cossenos: o cosseno
responde ao **ângulo**, e a componente do deslocamento paralela a $\mathbf{v}(s)$ não o
altera. Por essa razão a concordância verbal, com deslocamento maior, ainda se situa
acima da substituição `há`→`a` na tabela anterior.)

### As duas invariâncias, verificadas

**A permutação.** O código exibe $\cos = 1{,}000000$ para *O fogo atingiu o vale*
confrontada com *O vale atingiu o fogo*, resultado que corresponde à identidade derivada
no fundamento. A sentença embaralhada do corpus, contudo, resultou em $0{,}967$, e não em 1 —
divergência que exige explicação, dado que a identidade não admite exceção.

A causa é a caixa. A sentença de referência inicia-se por `Há`; a embaralhada apresenta
`há` em posição medial. Trata-se de **linhas distintas** da tabela, com
$\cos(\texttt{Há}, \texttt{há}) = 0{,}792$, de modo que os dois multiconjuntos não são
idênticos e a hipótese da identidade não se aplica. Uniformizando-se a caixa:

```
nativa uma incêndio Há mata de o área três atingiu dias.
   multiconjunto idêntico ao da base:  True
   cos = 1.000000
```

Exatamente 1, até a sexta casa decimal. A identidade é válida; o valor $0{,}967$
mensurava diferença tipográfica, e não reordenação.

**O desvio de grafia é eliminado do cálculo.** A forma `natvia` não consta entre as
20.000 formas da tabela: `has_vector = False` e norma $0$. A função filtra por
`t.has_vector`, donde

$$|s| : 12 \longrightarrow 11$$

O vetor **é** nulo, mas não ingressa na média: a forma defeituosa é **removida do
somatório e do denominador**. A média passa a incidir sobre outro objeto e sobre outro
número de termos, e o resultado ($0{,}992$) é elevado precisamente porque o token
problemático foi descartado antes de intervir. Fora da média, em comparação direta, o
vetor nulo produziria a indeterminação $0/0$ — interceptada pela guarda `if na and nb`,
que devolve `nan`. **Um desvio de grafia não torna a sentença menos próxima: retira a
forma do cálculo, sem qualquer indicação na saída.**


### A última linha da tabela

O par `ativo`×`extinto` resulta em apenas $0{,}263$, valor próximo ao de um par sem
relação. Trata-se dos dois extremos do campo `status_evento` da base, e a medida os trata
como praticamente não relacionados.

O resultado evidencia que o efeito não é simétrico nem previsível: antônimos situam-se
próximos **quando compartilham contexto** e distantes quando não o compartilham. Não é
possível corrigir a medida por regra de desconto de pares opostos, dado que o valor
numérico não permite determinar em qual dos dois casos se está.

### O custo da permutação, em uso

Em um relatório de eventos de fogo a identidade acima é particularmente grave: quase todo
campo da base ingressa na sentença como **argumento** de um verbo, e a permuta de dois
argumentos produz afirmação falsa que a medida não distingue da verdadeira. A atribuição
de papéis é precisamente a informação suprimida pela média.

### Quando usar

- **Para proximidade temática.** Agrupamento de documentos por assunto, sugestão de
  termos relacionados, recuperação do parágrafo de um texto que trate do mesmo tema que a
  consulta. É o que a medida efetivamente realiza, e o realiza adequadamente.
- **Como recurso de baixo custo e determinístico.** Consiste em consulta a tabela seguida
  de produto escalar. Não há modelo a executar nem *download* a realizar, e o mesmo par
  devolve invariavelmente o mesmo valor.
- **Para comparação de conjuntos de formas**, em que a ordem efetivamente não é
  pertinente — palavras-chave de um documento, campo semântico de um termo.

### Quando não usar

- **Para polaridade, negação ou oposição.** É o limite central: `quente`/`frio` e
  `sempre`/`nunca` são mensurados como próximos. Qualquer aplicação que dependa da
  distinção entre afirmação e negação, ou entre elogio e crítica, emprega instrumento
  inadequado.
- **Para qualquer aplicação dependente da ordem.** A média produz cosseno exatamente
  igual a $1$ entre uma sentença e sua permutação. Sujeito e objeto, voz ativa e voz
  passiva, atribuição de papéis: nada disso sobrevive à agregação.
- **Como medida de correção.** Um desvio de grafia é eliminado do cálculo sem
  sinalização; um desvio de concordância não altera vetor algum, dado que `dorme` e
  `dormem` são formas distintas com vetores próprios, mas a média se desloca de forma
  reduzida.
- **Com vetor estático, sob expectativa de desambiguação.** Há **uma única** linha por
  forma escrita. As acepções de `banco` como assento e como instituição financeira
  compartilham o mesmo vetor, que corresponde a uma média indistinta dos dois sentidos. A
  distinção exige vetores contextuais, o que constitui outra técnica.
- **Na comparação de cossenos entre modelos distintos.** O valor não possui escala
  absoluta: os $0{,}509$ de `incêndio`/`fogo` só adquirem significado em confronto com os
  $-0{,}063$ de `área`/`advérbio`, **na mesma tabela**. A substituição do modelo altera
  todos os valores.
